# Figure S6 — MS risk genes, from GWAS loci to the mature oligodendrocyte subtype (v16 FINAL)

**What the figure claims.**  MS GWAS signal reaches the oligodendrocyte lineage in a way that
depends on how the risk set is *defined at the gene level*.  Positional evidence (A: four loci
resolved to single genes by their lead SNP; B: how many platform genes fall in those windows by
chance) is followed by two gene-level landing heatmaps drawn on the same scale: the MS
MAGMA **top-1000** set restricted to genes measurable on the platform (D, 223 rows) and the
**cis-window** set of 38 genes (E).  D shows nothing that a reader can pick out — that is the
point — while E's set lands on MOL6 in both channels.  F and G then place the same contrast in
per-nucleus scores across three library chemistries and in the 74-trait cross-channel ranking.

| Panel | File | Size (cm) | Content |
|---|---|---|---|
| A | `figS6v16_A_loci.pdf` | 10.78 x 2.79 | four MS loci: gene TSS ticks within lead-SNP +/- 500 kb, hit gene in red, lead-SNP *P*, signed TSS distance |
| B | `figS6v16_B_perm.pdf` | 5.38 x 3.05 | window permutation of the hit count: 38/331 = 11.5 %, *P* = 0.277 (10,000 draws) |
| C | `figS6v16_C_umapbins.pdf` | 17.00 x 3.00 | bin-size robustness of the Fig 6B RNA UMAP (single cells / 15 / 25 / 40), one shared colour scale, aspect locked |
| D | `fig6v32_C_top1000_landing_s6v18D.pdf` | 7.45 x 5.75 | **MS top-1000, 223 measurable genes** x 6 subtypes, RNA + ATAC, per-gene z; **deliberately unnamed** |
| E | `fig6v31_C_landing_s6v18E.pdf` | 7.45 x 5.75 | **cis-window 38 genes** x 6 subtypes, same scale; the four genes resolved in A are named |
| – | `figS6v18_DE_legend_v.pdf` | 1.62 x 5.75 | the one legend shared by D and E, as a vertical column at the right edge of row 3 |
| F | `figS6v16_F_cellboxes.pdf` | 8.08 x 5.75 | per-nucleus MS scDRS score, OPC vs Oligo, 2 channels x 3 chemistries, MWU *P* per panel |
| G | `fig6D_heatmap_pair_compact_s6v16G.pdf` | 8.64 x 5.39 | the 74-trait expression/chromatin pair (this was Figure 6 D before the move) |

Delivered sheet: `figS6_out/Figure_S6_v16.pdf`, **17.00 x 18.91 cm**, four rows (heights 3.05 / 3.00 / 5.75 /
5.75 cm), page-width gate 17.00 cm, page-height soft gate 18.00 cm (hard cap 20.50). The sheet
is composed by the delivered driver outside this notebook (`assembly/figS6v16_run_all.sh`):
per the author instruction the **composition code is not part of the notebook** — the notebook
reproduces the seven panels and the shared legend at the delivered sizes, and the driver
keeps producing the page.

**Display vs tested quantity — read this before the numbers.**  The heatmaps are coloured by a
*per-gene z across the six subtypes*; the statistic that is tested, and that orders the rows, is
`Δ = mean(MOL6) − mean(COP/NFOL)`.  The z is an affine, per-gene transform of Δ, so colour depth is comparable **within** a
row and never **between** rows.  The tested contrast therefore still has to be stated in the
legend even though its column was deleted from the face.

---

## 0. Environment, paths and versions

The notebook reads **only delivered tables and small caches**; nothing is re-scored, re-tested or
re-permuted by the drawing code (house rule: every statistic on the face is read from a CSV that
`upstream/` produced, so the figure cannot disagree with the ledger).

| Variable | Default (in-repo) | Points at |
|---|---|---|
| `DSCI_WORK_ROOT` | three levels above `assembly/` | the analysis tree; needed only to *rebuild* the caches |
| `S6_OUT` | `figS6_out/` | where the panel PDFs are written |
| `S6_V8` | `tables_v16/` | delivered statistic tables (window permutation, MWU ledger) |
| `S6_SCMS` | `tables_v16/` | the locus hit table (panel A) |
| `S6_CACHE` | `figS6_out/caches/` | column projections of the big objects (GTF, h5ad, 6 GB score sweeps) |
| `P08C_TABLES` | `tables_v16/` | the P0_8C gene-level tables behind D and E |

With the caches present the notebook runs on a laptop with no access to the sequencing tree.
`upstream/S6v16_export_caches.py` regenerates them from `DSCI_WORK_ROOT`; the caches are
projections only — no statistic is computed in that script.

In [1]:
import os, sys, glob, time
import numpy as np, pandas as pd

REPO = os.path.abspath(os.getcwd())            # run the notebook from figS6/
ASSY = os.path.join(REPO, "assembly")
TABD = os.path.join(REPO, "tables_v16")
OUTD = os.path.join(REPO, "figS6_out")
CACH = os.path.join(OUTD, "caches")
for _d in (OUTD, CACH):
    os.makedirs(_d, exist_ok=True)

os.environ.setdefault("DSCI_WORK_ROOT", os.path.abspath(os.path.join(REPO, "..", "..")))
os.environ["S6_OUT"] = OUTD
os.environ["S6_V8"], os.environ["S6_SCMS"] = TABD, TABD
os.environ["S6_CACHE"] = CACH
os.environ["P08C_TABLES"] = TABD
sys.path.insert(0, ASSY); sys.path.insert(0, os.path.join(REPO, "upstream"))

# --- module pinning (a shared-kernel hazard that per-script processes never show) ---------
# The repo carries its own arial_style / two_tier_check / support_common / fig6_palette; the
# analysis tree has same-named files too (<work>/scripts/arial_style.py and
# .../fig6_repro/deps/arial_style.py; neither has use_two_tier). The delivered scripts each
# run in their own process with a fixed resolution order; the notebook shares one kernel, and
# support_common inserts .../fig6_repro/deps at sys.path[0], so later cells would pick the
# wrong copy. Pinning the repo copies into sys.modules here makes every later
# `from arial_style import ...` hit the cache, immune to path order.
import importlib
for _n in ("arial_style", "two_tier_check", "support_common", "fig6_palette"):
    if os.path.exists(os.path.join(ASSY, _n + ".py")):
        sys.modules.pop(_n, None)
        importlib.import_module(_n)
print("pinned   :", ", ".join("%s=%s" % (k, os.path.basename(sys.modules[k].__file__))
                              for k in ("arial_style", "two_tier_check", "support_common")
                              if k in sys.modules))
print("repo     :", REPO)
print("analysis :", os.environ["DSCI_WORK_ROOT"], "(only used if a cache is missing)")
print("caches   :", ", ".join(sorted(os.listdir(CACH))) or "NONE — panels fall back to the analysis tree")

pinned   : arial_style=arial_style.py, support_common=support_common.py
repo     : <repo>/figS6
analysis : <work> (only used if a cache is missing)
caches   : fig6_cellscore.csv, s6a_gene_coords.csv, s6c_umap_score.csv, s6f_10x_atac.csv, s6f_10x_rna.csv, s6f_HT_scCAT_seq_HT_atac.csv, s6f_HT_scCAT_seq_HT_rna.csv, s6f_dsciCAT_seq_atac.csv, s6f_dsciCAT_seq_rna.csv


### 0.1 Where the inputs come from

| File (in `tables_v16/` unless noted) | Produced by | Used by |
|---|---|---|
| `ms_locus_overlap_table.csv` | `upstream/ms_locus_overlap.py` (lead SNP +/- 500 kb against GENCODE v45) | A |
| `figS6v8_window_permutation{,_null}.csv` | `assembly/S6v8_window_permutation.py` (10,000 draws, seed fixed in the script) | B |
| `p08c_genelevel_all6_pool.csv`, `p08c_dual_celllevel223.npz` | `assembly/p0_8c_gene_enrichment.py` / `p0_8c_fig6c_full223.py` (scDRS gene-level scores per subtype, both channels) | D |
| `p08c_c1_rna_atac_by_subtype.csv`, `p08c_c2_celllevel38.npz` | `assembly/p0_8c_c1_cluster_probe.py` / `p0_8c_c2_stability.py` | E |
| `p08c_dual_setlevel_matched2d_mol6.csv` | `assembly/p0_8c_match2d.py` — **the single code path for every set-level P on D/E** (size-matched 2D permutation, 5x5 strata, 10,000 draws) | D, E, legend text |
| `figS6v10_G_opc_oligo_mwu.csv` | `upstream/S6v10_G_mwu_p.py` | F |
| `fig6D_delta1_74traits_per_tech.csv`, `fig6D_crosschem_ladder.csv` | `upstream/D6_crosschem_analysis.py` (same-nucleus scores of the three chemistries) | G |
| `caches/s6a_gene_coords.csv` | `upstream/S6v16_export_caches.py` | A |
| `caches/s6c_umap_score.csv` | same (UMAP_1/2 + cell_type + MS RNA score, 5,962 nuclei) | C |
| `caches/s6f_<job>.csv` | same (per-nucleus MS `norm_score` + label, 6 arms) | F |

The three chemistries are the deposited ones (CNGBdb **CNP0010263**): dsciCAT-seq (own),
10x Multiome, HT-scCAT-seq. The chromatin arms of F and G come from the **log1p-corrected**
re-run of 2026-11-04 (`upstream/README.md`, "Input transform"); the RNA arms are the delivered tree.

**Data origins — from which RDS/h5ad every number here comes.**  The own multiome traces to the
authors' RDS objects converted under `fig6/upstream/README.md` Step 0 (each verified bit-for-bit):
`FY/oligo/3k_dsci_oligo_subset.RDS` → `3k_dsci_oligo_subset.h5ad` (`rds_to_dump.R` + `build_h5ad.py`;
5,977 nuclei, the six Fig 2 subtypes, UMAP inherited — the panels' 5,962 = the paired subset that has
both-channel scores) and `FY/3k_dsci_ht_10x_gs_subset.RDS` → `genescore_oligo_5977.h5ad`
(`export_genescore.R` + `build_genescore_h5ad.py`; 24,333 ATAC gene scores, same nuclei, same order).
The per-nucleus MS `norm_score` trees behind C/F/G are `*.score.gz` / `PASS_*.full_score.gz` sweeps
run by `upstream/D5_run_scdrs.sh` (scdrs 1.0.2): the RNA arms are the delivered tree
(`D_crosschemistry/scdrs/`), the ATAC arms the **log1p-corrected** rerun (`scdrs_log1p_20261104/`).
The only external dataset is the mouse EAE **GSE250589** (panel F mouse rung, `M2b_formal_authoranno.py`).
The full production table is `figS6/upstream/README.md`.

### 0.2 Software versions

In [2]:
import matplotlib, scipy, fitz
print("python   ", sys.version.split()[0])
for m in (np, pd, scipy, matplotlib, fitz):
    print("%-10s %s" % (m.__name__, m.__version__))
print("PyMuPDF is load-bearing: every size claim below is measured from the produced PDF,")
print("never taken from the nominal canvas (tight bounding box != figsize).")

python    3.9.23
numpy      1.26.4
pandas     2.3.1
scipy      1.13.1
matplotlib 3.9.4
fitz       1.24.11
PyMuPDF is load-bearing: every size claim below is measured from the produced PDF,
never taken from the nominal canvas (tight bounding box != figsize).


### 0.3 Frozen criteria (fixed before the run, not chosen afterwards)

| | |
|---|---|
| tested contrast | `Δ = mean(MOL6) − mean(COP/NFOL)` per gene, per channel — the same poles as Fig 6 A/B, so the supplement and the main figure test one thing |
| display | per-gene z across the six subtypes: `z = (L − mean_s L) / sqrt(sd_s^2 + s0^2)`, `s0` = the row-scale floor (D: RNA 0.07755 / ATAC 0.02619; E: 0.1476 / 0.07267), clipped at ±2.0, one colour scale shared by RNA and ATAC and by D and E |
| detection floor | a cell is grey if the gene is detected in <10 % of that subtype's nuclei: D 255/1338 (RNA) and 81/1338 (ATAC) cells; E 49/228 and 7/228 |
| off scale | a white triangle at the clipped edge of the cell: **D 3 RNA + 11 ATAC = 14 cells, E 2 + 1 = 3 cells**, all of them in the COP/NFOL reference column (see §4) |
| set-level test | size-matched 2D permutation, 5x5 strata (n_hit x mean detection), 10,000 draws, one seed, one code path (`p08c_dual_setlevel_matched2d_mol6.csv`): top-1000 P = 0.096 (RNA) / 0.745 (ATAC); cis38 P = 0.0017 / 0.0007; myelin anchor 0.0107 / 0.0080; 38 vs the 293 non-window platform genes 0.0127 / 0.0066 |
| row order | by \|Δ\| descending — display only, never a test |
| naming | D is **not** named (a null panel with named rows invites readers to find a pattern; and a name list produced by \|Δ\| plus a row-spacing rule changes when the page height changes). E names exactly the four genes that panel A resolves to single loci — positional evidence that is not on the face of E |
| jitter / RNG | panel F dots: `default_rng(20261117)`; panel C hexbins: deterministic |
| typography | two sizes only, 8.0 pt (letters, axis titles) and 6.0 pt (everything else); Arial; text on the face is black except subtype labels and arrows, which keep the identity colours |

The page geometry is **solved, not guessed**: `PW = 1.78 + 12·CW` and `2·PW + 2·0.24 + LW = 17.00`,
with `LW` measured from the legend's own ink — the same arithmetic as `assembly/figS6v16_run_all.sh`.

---

## 1. Row 1 — panels A and B: positional evidence

**A — four loci.**  For each of PHLDB1 (chr11), PHGDH (chr1), BCAS1 (chr20) and PLLP (chr16) the
lead SNP is plotted with every GENCODE gene TSS inside ±500 kb; the hit gene is the red tick, and
the signed distance from lead SNP to gene TSS is printed in the corner. These four are the genes
that the window analysis resolves to a *single* locus, which is why they — and only they — are
named in panel E.

**B — is 38/331 more than chance?**  The number of platform genes inside the lead-SNP windows of
the MS hit list, permuted 10,000 times against size-matched random windows: observed 38 of 331
(11.5 %), permutation *P* = 0.277. The positional evidence is therefore *locus-level*, not
*count-level* — which is exactly why the supplement then asks the gene-level question in D and E.

Both panels come from `assembly/figS6v16_panels.py`; the file also draws C and F, so its header
(the two-size style, the ink-measuring `savefig`, the colour constants) is executed here.

In [3]:
# ------------------------------------------------------------ assembly/figS6v16_panels.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/figS6v16_panels.py")
# -*- coding: utf-8 -*-
"""FigS6 v16 panel re-render (three rows, six cells, 8.38 cm wide each)
====================================================
Per FIG6_S6_LAYOUT_FINAL_20261117.md §4.1:
  row 1  A four MS loci tracks  | B 331-platform-gene window permutation
  row 2  C top-1000 evidence chain (223 rows) | D 38-gene landing  <- produced by p0_8c_fig6c_{top1000,landing}.py via env
  row 3  E RNA per-cell norm_score, three chemistries | F ATAC per-cell norm_score, three chemistries
This script only renders A/B/E/F (C/D reuse the main figure's data chain, so a second implementation cannot drift the numbers).

Discipline (figure-house-spec):
  * Two font sizes only: 8.0 (panel letters / axis titles / row headers) and 6.0 (ticks, corner notes, box names, statistics).
    v10's 5.5/5.7/5.8/6.2/6.8 sizes were all raised; crowding is never solved by shrinking type (spec §3.3: breathe first, wrap second, trim last).
  * All face text is pure black (v10's 0.2/0.35/0.45 grey and red/blue emphasis text was blackened; emphasis colour stays on the marks themselves).
  * Panel letters are added by the compose layer; no title sentence inside a cell.
Usage: DSCI_WORK_ROOT=<work> python3 figS6v16_panels.py
"""
import os, sys, glob
import numpy as np, pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("DSCI_WORK_ROOT", os.path.abspath(os.path.join(HERE, "..", "..", "..")))
HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE); sys.path.insert(0, WORK + "/scripts")
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "upstream"))   # repo layout: ../upstream
from support_common import R, say
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import fitz
from arial_style import use_arial, patch_fitz_arial
use_arial({"axes.linewidth": 0.5, "savefig.dpi": 600})
patch_fitz_arial(fitz)

# ---- portability contract (the repo notebook and the local pipeline share one code path; only env vars differ) ----
#   S6_OUT   panel PDF output dir       default = analysis tree .../P0_8C/panels (historical behaviour kept)
#   S6_V8    delivered tables_v8       default = .../SUPPORT_v2/tables_v8
#   S6_CACHE small re-input caches (gene coords / UMAP table / per-nucleus scores) - the run also works without them,
#            it just has to recompute from GTF / h5ad / 6 GB .score.gz; results are written back to the cache.
#   S6_SCMS  scDRS output tree (locus hit table + GENCODE GTF)
PAN = os.environ.get("S6_OUT", R + "/P0_8C/panels"); os.makedirs(PAN, exist_ok=True)
SUP2 = R + "/SUPPORT_v2"; V8 = os.environ.get("S6_V8", SUP2 + "/tables_v8")
CACHE = os.environ.get("S6_CACHE", os.path.join(PAN, "..", "caches")); os.makedirs(CACHE, exist_ok=True)
SCMS = os.environ.get("S6_SCMS", WORK + "/FY/oligo/scdrs_ms_out")
R_OUT = R
CM = 2.54
FS_LAB, FS_TITLE = 6.0, 8.0                      # the two-size policy
RED, OLIGO, OPC = "#D51F26", "#1F77B4", "#FF7F0E"
ARMS = ["dsciCAT-seq (own)", "10x Multiome", "HT-scCAT-seq"]
ARM_LAB = {"dsciCAT-seq (own)": "dsciCAT-seq", "10x Multiome": "10x Multiome",
           "HT-scCAT-seq": "HT-scCAT-seq"}
W_CELL = 8.38
BLUERED = mcolors.LinearSegmentedColormap.from_list("bluered", ["#9ECAE1", "#FFFFFF", "#E60000"])


def tw(t, fs=FS_LAB):
    """Arial(helv) text width in cm: wrap/trim decisions are always measured, never estimated."""
    return fitz.get_text_length(t, fontname="helv", fontsize=fs) * CM / 72.0


def savefig(fig, name):
    out = os.path.join(PAN, name + ".pdf")
    fig.savefig(out, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    d = fitz.open(out); p = d[0]; C = CM / 72
    w, h = p.rect.width * C, p.rect.height * C
    ws = p.get_text("words")
    right = max((x[2] * C for x in ws), default=0); bot = max((x[3] * C for x in ws), default=0)
    # fitz 1.24 span["color"] is an sRGB int (not a tuple): only pure black/white allowed on the face
    cols = sorted({s["color"] for bl in p.get_text("dict")["blocks"]
                   for ln in bl.get("lines", []) for s in ln["spans"] if s["text"].strip()})
    nonbw = ["#%06x" % c for c in cols if c not in (0x000000, 0xffffff)]
    szs = sorted({round(s["size"], 2) for bl in p.get_text("dict")["blocks"]
                  for ln in bl.get("lines", []) for s in ln["spans"] if s["text"].strip()})
    d.close()
    print("  %-30s %.2f x %.2f cm | ink right %.2f/%.2f bottom %.2f/%.2f | sizes %s | non-bw %s"
          % (name + ".pdf", w, h, right, w, bot, h, szs, nonbw))
    assert right <= w - 0.01 and bot <= h - 0.01, name
    return out



# ---- section A of the delivered file ----
def panel_A():
    import ms_locus_overlap as mlo
    HIT = pd.read_csv(SCMS + "/ms_locus_overlap_table.csv")
    gc_ = os.path.join(CACHE, "s6a_gene_coords.csv")
    if os.path.exists(gc_):                       # the cache stores only (chrom, tss, name)
        _g = pd.read_csv(gc_)
        GENES = list(zip(_g.chrom.astype(str), _g.tss.astype(int), _g.name.astype(str)))
        say("[A] gene coordinates from cache %s (%d entries)" % (os.path.basename(gc_), len(GENES)))
    else:
        GENES = mlo.load_genes(SCMS + "/gencode.v45.basic.annotation.gtf.gz")
        pd.DataFrame({"chrom": [g[0] for g in GENES], "tss": [g[1] for g in GENES],
                      "name": [g[2] for g in GENES]}).to_csv(gc_, index=False)
    SEL4 = ["PHLDB1", "PHGDH", "BCAS1", "PLLP"]
    fig = plt.figure(figsize=(11.36 / CM, 3.40 / CM))   # 3.40 canvas -> 2.75 ink after tight, matching B's 2.73 height
    # Keeps the original S6 row-1 layout: A full 11.36 + B 5.40 (a histogram never needs half a page), both cells 3.00 high.
    # Top margin 1.15: two title lines + two in-cell corner-note lines; the second title line (chr11) would touch the first corner line (GWAS) by 0.04 cm, measured.
    gs = fig.add_gridspec(1, 4, left=0.055, right=0.995, top=1 - 1.15 / 3.40, bottom=0.42 / 3.40,
                          wspace=0.16)
    for k, gname in enumerate(SEL4):
        h = HIT[HIT.gene == gname].iloc[0]
        ax = fig.add_subplot(gs[0, k])
        ch = str(h["chr"]); lo, hi = int(h.snp_pos) - 500_000, int(h.snp_pos) + 500_000
        for c2, t, nm in GENES:
            if c2 == ch and lo <= t <= hi and nm.upper() != gname:
                ax.plot([t, t], [0.42, 0.58], color="grey", lw=0.8)
        ax.plot([h.gene_tss, h.gene_tss], [0.42, 0.58], color=RED, lw=2.0)
        ax.plot([h.snp_pos, h.snp_pos], [0.70, 0.86], color="black", lw=1.2)
        ax.plot([h.gene_tss, h.gene_tss], [0.40, 0.28], color=RED, lw=0.5)
        ha = "left" if h.gene_tss > hi - 0.18 * (hi - lo) else (
            "right" if h.gene_tss < lo + 0.18 * (hi - lo) else "center")
        ax.text(h.gene_tss, 0.17, gname, fontsize=FS_LAB, ha=ha, va="center", color="black")
        # Two-line titles: a single "PHLDB1  chr11" line @8pt measures 1.85 cm > facet width 1.79 cm and would collide
        ax.set_title("%s\nchr%s" % (gname, ch), fontsize=FS_TITLE, pad=1.2, linespacing=1.25)
        ax.text(0.02, 0.93, "GWAS lead-SNP\nP = %.0e" % h.pvalue, transform=ax.transAxes,
                ha="left", va="top", fontsize=FS_LAB, linespacing=1.3, color="black")
        ax.text(0.02, 0.66, "dist %+.0f kb" % (h.distance_bp / 1000.0), transform=ax.transAxes,
                ha="left", va="top", fontsize=FS_LAB, color="black")
        ax.set_xlim(lo, hi); ax.set_ylim(0, 1.0)
        ax.set_xticks([lo, hi])
        ax.set_xticklabels(["%.1f" % (lo / 1e6), "%.1f" % (hi / 1e6)], fontsize=FS_LAB)
        for lb, al in zip(ax.get_xticklabels(), ["left", "right"]):
            lb.set_ha(al)
        ax.set_yticks([])
        for sp in ["top", "right", "left"]:
            ax.spines[sp].set_visible(False)
        for sp in ax.spines.values():
            sp.set_linewidth(0.5)
        ax.tick_params(width=0.5, length=2, pad=0.8)
    return savefig(fig, "figS6v16_A_loci")



panel_A()

[22:50:54] [A] gene coordinates from cache s6a_gene_coords.csv (13402 entries)
  figS6v16_A_loci.pdf            10.78 x 2.79 cm | ink right 10.73/10.78 bottom 2.74/2.79 | sizes [6.0, 8.0] | non-bw []


'<repo>/figS6/figS6_out/figS6v16_A_loci.pdf'

In [4]:
# ------------------------------------------------------------ assembly/figS6v16_panels.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/figS6v16_panels.py")
# ---- section B of the delivered file ----
CM = 2.54   # shared-kernel hygiene: the D/E panel cells overwrite CM with 1/2.54 (cm->inch inverse); re-asserted per section, a no-op when the whole file runs
def panel_B():
    WPERM = pd.read_csv(V8 + "/figS6v8_window_permutation.csv").set_index("stat")
    WNULL = pd.read_csv(V8 + "/figS6v8_window_permutation_null.csv")
    fig, ax = plt.subplots(figsize=(5.40 / CM, 3.00 / CM))
    fig.subplots_adjust(left=0.175, right=0.985, top=0.90, bottom=0.235)
    obs_pct = float(WPERM.loc["observed (%)", "value"])
    ax.hist(WNULL.null_pct.values, bins=np.arange(6, 17, 0.5), color=(0.69, 0.769, 0.871),
            edgecolor=(0.28, 0.40, 0.56), linewidth=0.4, zorder=3)
    ax.axvline(obs_pct, color=RED, lw=1.1, zorder=4)
    ax.set_xlim(5.4, 16.6); ax.set_xticks([6, 8, 10, 12, 14, 16])
    # A single 8pt line measures 4.9 cm > the 4.4 cm available and would run past the page right (the self-check caught 5.49>5.38) -> two-line titles, no font shrink
    ax.set_xlabel("% of the 331 platform genes\ninside window", fontsize=FS_TITLE,
                  labelpad=2.0, linespacing=1.3)
    ax.set_ylabel("permutations", fontsize=FS_TITLE, labelpad=2.0)
    ax.tick_params(labelsize=FS_LAB, length=2, width=0.5, pad=1.0)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    pperm = float(WPERM.loc["permutation P (one-sided, observed vs null)", "value"])
    # The corner note sits right of the zero axis and above the first category, padded with white (the text-on-data gate only checks points and lines, not bars, so this must be dodged by hand)
    ax.text(0.985, 0.965, "obs 38/331 = %.1f%%\nn = 10,000;  P = %.3f" % (obs_pct, pperm),
            transform=ax.transAxes, fontsize=FS_LAB, va="top", ha="right", linespacing=1.35,
            color="black", bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=1.0),
            zorder=6)
    return savefig(fig, "figS6v16_B_perm")



panel_B()

  figS6v16_B_perm.pdf            5.38 x 3.05 cm | ink right 5.26/5.38 bottom 3.00/3.05 | sizes [6.0, 8.0] | non-bw []


'<repo>/figS6/figS6_out/figS6v16_B_perm.pdf'

---

## 2. Row 2 — panel C: the UMAP behind Fig 6B, at four bin sizes

**Data chain**: `obs.UMAP_1/2` + `cell_type` of the lineage object joined to the per-nucleus MS
RNA score (`caches/s6c_umap_score.csv`, 5,962 nuclei). Panel C exists because a reader should not
have to trust one rendering choice: the same nuclei are hex-binned at single-cell resolution and
at gridsize 15, 25 (the Fig 6B choice) and 40, on one shared colour scale, with the aspect locked
to the data's own length-to-width ratio (1.63) so no tile is stretched.

Nothing about the embedding is re-derived here, and no conclusion depends on its shape.

In [5]:
# ------------------------------------------------------------ assembly/figS6v16_panels.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/figS6v16_panels.py")
# ---- section C of the delivered file ----
CM = 2.54   # shared-kernel hygiene: the D/E panel cells overwrite CM with 1/2.54 (cm->inch inverse); re-asserted per section, a no-op when the whole file runs
# Original S6 row-2 layout: RNA in four cells (single cells / gridsize 15 / 25 / 40) + one colour bar on the right, full width 17.00.
# Only two compliance fixes vs the original: (1) set_aspect("equal") locks the shape (the original's 3.55x3.20 flattened the 1.63
# UMAP); (2) all type 8.0/6.0, bold and grey text removed, annotation colour kept on marks only.
def panel_C():
    import anndata as ad
    CW, CH = 17.00, 3.00
    fig = plt.figure(figsize=(CW / CM, CH / CM))
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)      # exact cm placement, no tight bbox
    cc_ = os.path.join(CACHE, "s6c_umap_score.csv")
    if os.path.exists(cc_):                           # the four columns are this panel's entire input
        dU = pd.read_csv(cc_)
        print("  [C] UMAP+score from cache %s" % os.path.basename(cc_))
    else:
        obs = ad.read_h5ad(WORK + "/FY/oligo/3k_dsci_oligo_subset.h5ad", backed="r").obs
        CT = pd.read_csv(R + "/O1_own_multiome_paired/tables/cell_level_table.csv", low_memory=False)
        dU = obs[["UMAP_1", "UMAP_2", "cell_type"]].join(
            CT.set_index("cell_id")[["PASS_Multiple_sclerosis_rna"]], how="inner")
        dU = dU.astype({"UMAP_1": float, "UMAP_2": float, "PASS_Multiple_sclerosis_rna": float})
        dU["cell_type"] = dU["cell_type"].astype(str)     # cell_type is categorical; astype(float) on the whole table would blow up
        dU.to_csv(cc_, index=False)
    say = lambda m: print("  " + m)
    say("[C] UMAP reconciled on %d nuclei" % len(dU))
    xr = max(np.nanpercentile(dU.PASS_Multiple_sclerosis_rna, 99),
             abs(np.nanpercentile(dU.PASS_Multiple_sclerosis_rna, 1)))
    x0, x1 = np.percentile(dU.UMAP_1, [0.2, 99.8]); y0, y1 = np.percentile(dU.UMAP_2, [0.2, 99.8])
    ASP = (x1 - x0) / (y1 - y0)
    TW, TY = 3.55, 0.46
    TH = TW / ASP                                   # locked aspect: cell height from the data's own length/width ratio
    TX = [0.62, 4.52, 8.42, 12.32]
    TIT = ["single cells", "gridsize 15", "gridsize 25 (Fig 6B)", "gridsize 40"]
    def tile(i):
        a = fig.add_axes([TX[i] / CW, 1 - (TY + TH) / CH, TW / CW, TH / CH])
        a.set_title(TIT[i], fontsize=FS_TITLE, pad=1.8, loc="left")
        a.set_xticks([]); a.set_yticks([]); a.set_xlim(x0, x1); a.set_ylim(y0, y1)
        a.set_xlabel("UMAP_1", fontsize=FS_LAB, labelpad=1.2)
        a.set_aspect("equal")
        for sp in a.spines.values(): sp.set_linewidth(0.5)
        return a
    ax1 = tile(0)
    ax1.scatter(dU.UMAP_1, dU.UMAP_2, s=0.35, c=dU.PASS_Multiple_sclerosis_rna, cmap=BLUERED,
                vmin=-xr, vmax=xr, linewidths=0, rasterized=True)
    HB = []
    for i, gs_ in ((1, 15), (2, 25), (3, 40)):
        a = tile(i)
        hb = a.hexbin(dU.UMAP_1, dU.UMAP_2, C=dU.PASS_Multiple_sclerosis_rna,
                      reduce_C_function=np.mean, gridsize=gs_, mincnt=1, cmap=BLUERED,
                      linewidths=0, edgecolors="none", zorder=2)
        hb.set_clim(-xr, xr); hb.set_rasterized(True); HB.append(hb)
    sp1 = dU.UMAP_1.max() - dU.UMAP_1.min(); sp2 = dU.UMAP_2.max() - dU.UMAP_2.min()
    mol = dU[dU.cell_type == "MOL6"][["UMAP_1", "UMAP_2"]].median()
    cop = dU[dU.cell_type == "COP/NFOL"][["UMAP_1", "UMAP_2"]].median()
    # The original ha="center" let the arrow pierce the middle of the text box (text-on-data gate, LINE-ACROSS-TEXT 22%).
    # Now anchored at the box's lower-left corner + white pad: the leader ends at the box corner, shrinkB backs off another 2 pt. Offsets compensate half a cell per label, by measured width.
    WB = dict(facecolor="white", edgecolor="none", alpha=0.85, pad=0.6)
    for a in (ax1, HB[0].axes):
        a.annotate("MOL6", xy=(mol.UMAP_1, mol.UMAP_2),
                   xytext=(mol.UMAP_1 + 0.02 * sp1, mol.UMAP_2 + 0.26 * sp2), fontsize=FS_LAB,
                   color=RED, ha="left", va="bottom", bbox=WB,
                   arrowprops=dict(arrowstyle="->", color=RED, lw=0.7, shrinkA=1, shrinkB=2))
        a.annotate("COP/NFOL", xy=(cop.UMAP_1, cop.UMAP_2),
                   xytext=(cop.UMAP_1 + 0.44 * sp1, cop.UMAP_2 + 0.30 * sp2), fontsize=FS_LAB,
                   color="#084594", ha="center", va="bottom", bbox=WB,
                   arrowprops=dict(arrowstyle="->", color="#084594", lw=0.7, shrinkA=1, shrinkB=2))
    cbax = fig.add_axes([16.05 / CW, 1 - (TY + TH) / CH, 0.30 / CW, TH / CH])
    cbR = matplotlib.colorbar.ColorbarBase(cbax, cmap=BLUERED,
                                           norm=matplotlib.colors.Normalize(-xr, xr))
    cbR.outline.set_linewidth(0.4)
    cbR.ax.tick_params(labelsize=FS_LAB, length=1.8, width=0.4, pad=1.0)
    cbR.set_label("norm_score", fontsize=FS_LAB, labelpad=1.5)
    out = os.path.join(PAN, "figS6v16_C_umapbins.pdf")
    fig.savefig(out); plt.close(fig)
    d = fitz.open(out); p = d[0]; C = CM / 72
    ws = p.get_text("words")
    szs = sorted({round(s["size"], 2) for bl in p.get_text("dict")["blocks"]
                  for ln in bl.get("lines", []) for s in ln["spans"] if s["text"].strip()})
    cols = sorted({s["color"] for bl in p.get_text("dict")["blocks"]
                   for ln in bl.get("lines", []) for s in ln["spans"] if s["text"].strip()})
    print("  %-30s %.2f x %.2f cm | ink right %.2f/%.2f | sizes %s | non-bw %s"
          % ("figS6v16_C_umapbins.pdf", p.rect.width * C, p.rect.height * C,
             max(x[2] * C for x in ws), p.rect.width * C, szs,
             ["#%06x" % c for c in cols if c not in (0x000000, 0xffffff)]))
    d.close()
    return out



panel_C()

  [C] UMAP+score from cache s6c_umap_score.csv
  [C] UMAP reconciled on 5962 nuclei
  figS6v16_C_umapbins.pdf        17.00 x 3.00 cm | ink right 16.94/17.00 | sizes [6.0, 8.0] | non-bw ['#074594', '#d51f26']


'<repo>/figS6/figS6_out/figS6v16_C_umapbins.pdf'

---

## 3. Row 3 — geometry is solved before D and E are drawn

Row 3 carries three columns: D, E and the shared vertical legend. The legend used to be its own
row (a horizontal band, 1.05 cm plus a 0.15 cm attachment gap); making it a column of the same row
recovered **1.20 cm of page height** (20.11 → 18.91 cm) at the cost of 14 % of the subtype-column
width. Row height, cell pitch and both font sizes did not move — house spec ranks added whitespace
above shrinking type, and shrinking type is the one thing that is not allowed.

The width is not a rounded guess. `PW = 1.78 + 12·CW` (gene-name column 1.54 + right pad 0.24,
twelve subtype columns) and `2·PW + 2·0.24 + LW = 17.00`, where `LW` is the legend's own width
estimate + 0.11 — the same number the delivered `figS6v16_run_all.sh` reads off the legend
script's stdout. The cell below measures it, solves the equation, and writes the result into the
environment that D, E and the legend read.

In [6]:
# measure pass: no LEG_W in the environment yet -> the script uses its own default canvas
os.environ.pop("LEG_W", None)
# ------------------------------------------------------------ assembly/figS6v18_legend_right.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/figS6v18_legend_right.py")
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""figS6v18_legend_right.py - the VERTICAL legend shared by D/E (author 2026-11-17: DE shift left overall, the legend squeezed into the rightmost column).

Why vertical: the horizontal shared legend band (figS6v17, 17.00 x 1.05) took a whole row, page height 20.11 cm.
The vertical version joins row 3 as its third column, saving 1.05 row height + 0.15 attachment gap = **1.20 cm**.
The price: D/E give up this column's width, so the subtype-column width CW shrinks from 0.55 to ~0.48 (the body still has 6 columns x 2 blocks,
cells only narrow 13%, row height and type size unchanged - house spec ranks added whitespace above shrinking type).

The width is not guessed: every text is measured with fitz at its real width, the longest item fixes LW, then CW is solved back,
so that D + E + 2 x GAP + LW is **exactly 17.00** (the same idea as spreading Fig6's column gaps, SKILL §17.9).

Two font sizes 8.0/6.0; face text pure black (the colour-bar end numbers too); gradient direction conventional (blue− left/bottom, red+ right/top).
"""
import os
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import fitz

HERE = os.path.dirname(os.path.abspath(__file__))
# Default = three levels above the repo GitHub/figS6/assembly (the analysis-tree root); override with DSCI_WORK_ROOT.
WORK = os.environ.get("DSCI_WORK_ROOT", os.path.abspath(os.path.join(HERE, "..", "..", "..")))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from arial_style import use_arial, use_two_tier

PAN = os.environ.get("S6_OUT", WORK + "/figures/routeA_out/P0_8C/panels"); os.makedirs(PAN, exist_ok=True)
CM = 2.54
BLACK, GOLD, GREY, EDGE = "#000000", "#b8860b", "#d9d9d9", "#333333"
ZCLIP = float(os.environ.get("PANEL_ZCLIP", 2.0))
LW = float(os.environ.get("LEG_W", 2.00))          # canvas width (>= what is needed; tight trims to the ink width)
LH = float(os.environ.get("LEG_H", 5.75))          # matches the D/E row height so they share a top-aligned row
YCENTER = float(os.environ.get("LEG_YC", 3.38))    # the stack's vertical centre: aligns with the body mid-line (1.35+5.41)/2 of both cells
CMAP = LinearSegmentedColormap.from_list("by", ["#2166ac", "#f7f7f7", "#b2182b"])
use_arial(); use_two_tier()


def tw(t, fs=6.0):
    return fitz.get_text_length(t, fontname="helv", fontsize=fs) * CM / 72.0


# At STEP 0.36 a two-line 6pt key (0.42 together) would touch itself; raised to 0.44; stack height 3.97,
# still centred on the body mid-line 3.38 (1.40-5.37, inside the 5.75 page).
BARW, SWH, GAPX, STEP = 0.22, 0.15, 0.07, 0.44
# The off-scale key's swatch size and triangle: MS_OFF is the same 1.2 as the body triangles in the panel scripts,
# not "a bit bigger in the legend" - enlarging would lie: a reader measuring the legend against the face could not match the cells.
CHW, CHH, MS_OFF = 0.16, 0.11, 1.2
TITLE, TBAR = "z (per gene)", 1.60
KEYS = [("sw", "not detected"), ("car", "off scale"),
        ("bar", "tested\n(MOL6)"), ("bar", "reference\n(COP/NFOL)")]
# per-item width = swatch + gap + the widest text line
WID = {"sw": BARW + GAPX + tw("not detected"), "car": 0.34 + GAPX + tw("off scale"),
       "bar": None}
wbar = [BARW + GAPX + max(tw(l) for l in t.split("\n")) for k, t in KEYS if k == "bar"]
wcol = BARW + GAPX + tw("+%.1f" % ZCLIP)
tot = max(wcol, wbar[0], wbar[1], tw(TITLE), BARW + GAPX + tw("not detected"),
          0.34 + GAPX + tw("off scale"))
print("legend needs width %.2f cm (longest item = %s)" % (tot, ["colour scale", "not detected", "off scale",
      "tested(MOL6)", "reference(COP/NFOL)", "title"][
          [wcol, BARW + GAPX + tw("not detected"), 0.34 + GAPX + tw("off scale"),
           wbar[0], wbar[1], tw(TITLE)].index(tot)]))
if LW < tot + 0.10:
    raise SystemExit("LEG_W=%.2f does not fit the longest item %.2f (needs >= %.2f)" % (LW, tot, tot + 0.10))

fig = plt.figure(figsize=(LW / CM, LH / CM))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, LW); ax.set_ylim(0, LH); ax.axis("off")
X0 = 0.10
stack = 0.21 + 0.10 + TBAR + 0.30 + STEP * len(KEYS)      # title + colour scale + gap + four keys
y = YCENTER + stack / 2.0                                   # stack from the top down
ax.text(X0, y - 0.21, TITLE, fontsize=6.0, ha="left", va="bottom", color=BLACK)
y -= 0.21 + 0.10
ax.imshow(np.linspace(-ZCLIP, ZCLIP, 64)[:, None],
          extent=(X0, X0 + BARW, y - TBAR, y), cmap=CMAP, aspect="auto", zorder=3)
ax.add_patch(Rectangle((X0, y - TBAR), BARW, TBAR, fc="none", ec=EDGE, lw=0.3))
for dy, lab in ((0.0, "+%.1f" % ZCLIP), (TBAR / 2.0, "0"), (TBAR, "−%.1f" % ZCLIP)):
    ax.text(X0 + BARW + GAPX, y - dy, lab, fontsize=6.0, ha="left", va="center", color=BLACK)
y -= TBAR + 0.30
for kind, t in KEYS:
    yy = y - STEP / 2.0
    if kind == "sw":
        ax.add_patch(Rectangle((X0, yy - SWH / 2), BARW, SWH, fc=GREY, ec="#8c8c8c", lw=0.3))
    elif kind == "car":
        # The key must look exactly like the face: a WHITE triangle ms=1.2 stuck to the off-scale cell's left edge (negative) / right edge (positive),
        # on the saturated blue/red clipped at ±ZCLIP. This used to draw a 2.2 pt blue/red triangle -
        # matching the face in neither colour nor size (a reader hunting for blue triangles finds none).
        # Now a "swatch + same-size white triangle": swatch width 0.16, the two together still 0.34, so the legend width is unchanged and CW needs no re-solving.
        for k, (cx, fc, mk, mx) in enumerate((
                (X0, CMAP(0.0), "<", 0.030),                 # negative off-scale: blue swatch + white < on the left edge
                (X0 + 0.18, CMAP(1.0), ">", 0.130))):        # positive off-scale: red swatch + white > on the right edge
            ax.add_patch(Rectangle((cx, yy - CHH / 2), CHW, CHH, fc=fc, ec="none"))
            ax.plot(cx + mx, yy, marker=mk, ms=MS_OFF, color="#ffffff", mew=0.3, zorder=6)
    else:
        ax.add_patch(Rectangle((X0, yy - 0.038), BARW, 0.075,
                               fc=GOLD if "tested" in t else BLACK, ec="none"))
    lines = t.split("\n")
    for k, ln in enumerate(lines):        # two-line keys: centred on y as a block
        ax.text(X0 + (BARW if kind != "car" else 0.34) + GAPX,
                yy + (0.09 if len(lines) > 1 and k == 0 else (-0.09 if len(lines) > 1 else 0.0))
                - 0.0, ln, fontsize=6.0, ha="left", va="center", color=BLACK)
    y -= STEP
print("  stack %.2f-%.2f, centre %.2f (target %.2f)" % (y, y + stack, y + stack / 2.0, YCENTER))
out = PAN + "/figS6v18_DE_legend_v.pdf"
fig.savefig(out, dpi=600)
d = fitz.open(out); p = d[0]; ws = p.get_text("words")
sp = [s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]]
print("  %-30s page %.2f x %.2f cm | text right %.2f bottom %.2f top %.2f | sizes %s | non-bw %s"
      % (os.path.basename(out), p.rect.width * CM / 72, p.rect.height * CM / 72,
         max(w[2] * CM / 72 for w in ws), max(w[3] * CM / 72 for w in ws),
         min(w[1] * CM / 72 for w in ws), sorted({round(s["size"], 1) for s in sp}),
         [{"#%06x" % s["color"]} for s in sp if s["color"] not in (0, 16777215)]))
d.close()

legend needs width 1.51 cm (longest item = reference(COP/NFOL))
  stack 1.40-5.37, centre 3.38 (target 3.38)
  figS6v18_DE_legend_v.pdf       page 2.00 x 5.75 cm | text right 1.61 bottom 4.32 top 0.36 | sizes [6.0] | non-bw []


In [7]:
# The delivered recipe reads the legend's OWN width estimate off its stdout ("legend needs width ... cm")
# and takes LW = estimate + 0.11 (figS6v16_run_all.sh step 2). The legend cell just ran in this
# kernel, so its module-level `tot` is in the namespace. Re-measuring the PDF ink instead would
# add the X0=0.10 left margin (ink = X0 + tot) and give LW=1.72 - a geometry 0.05 cm wider per
# D/E column than the delivered page.
ink = tot
LW = round(ink + 0.11, 2)
CW = (17.00 - 2 * 0.24 - LW - 3.56) / 24.0
PW = 1.78 + 12 * CW
assert abs(2 * PW + 2 * 0.24 + LW - 17.00) < 0.02, (PW, LW)
for k, v in dict(PANEL_W="%.4f" % PW, PANEL_CW="%.4f" % CW, LEG_W="%.2f" % LW,
                 PANEL_H="5.75", PANEL_BODY_B="1.35", PANEL_ZMODE="1", PANEL_LEGEND="0",
                 PANEL_TITLE="0", PANEL_DCOLUMN="0", PANEL_CONTRAST="mol6").items():
    os.environ[k] = v
print("legend estimate %.2f -> LW %.2f | CW %.4f -> PW %.2f | row width %.2f (= 17.00)"
      % (ink, LW, CW, PW, 2 * PW + 2 * 0.24 + LW))

legend estimate 1.51 -> LW 1.62 | CW 0.4725 -> PW 7.45 | row width 17.00 (= 17.00)


## 4. Panel D — the MS top-1000 lands nowhere a reader can see

223 genes: the MS MAGMA top-1000, restricted to genes measurable on this platform (mapped minus
near-zero), each drawn across the six subtypes in both channels. The tested result is
*P* = 0.096 (RNA) and 0.745 (ATAC) against size-matched background sets — a null. The panel is
deliberately **not** labelled: a null panel with ten gene names on it invites the reader to hunt
for a pattern that the test says is not there, and a name list produced by |Δ| plus a minimum
row spacing changes whenever the page height changes (three geometries of the same data gave
8/10/10 names with only two in common).

**Why every off-scale triangle sits in the reference column.**  Under the per-gene z the
COP/NFOL column is the row-offset column: `L[:,0] ≡ 0` while the row mean is set by the other five
columns, so its |z| is systematically the largest (mean |z| 0.85/1.01 in the reference column vs
0.49/0.47 elsewhere). That is a property of the transform, not of the data — and it is *not* a
systematic colour bias: only 18 % (RNA) / 34 % (ATAC) of reference-column cells are negative, so
both directions occur. An alternative centring (five columns, reference excluded) was measured and
rejected: the reference column's mean |z| rises to 1.42/2.16 and the off-scale cells go from
3/11 to 54/86, i.e. the column becomes a stripe of triangles.

Historical note, kept because it is the kind of bug a gate cannot see: the triangles were once
drawn only when `abs(v) > HI and j > 0`, a leftover from the era when the reference column was a
constant-zero placeholder. Under the z display every off-scale cell *is* in column 0, so the
condition silently swallowed all of them and the legend key became a promise nothing fulfilled.
The guard is gone; the panel scripts now count what they draw and print it.

In [8]:
# run recipe for this panel (the same env the delivered driver exports)
os.environ["PANEL_TAG"] = "_s6v18D"
os.environ["PANEL_NAMES"] = "0"        # D stays unnamed (a null panel must not hand readers a pattern to hunt for)
# ------------------------------------------------------------ assembly/p0_8c_fig6c_top1000.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/p0_8c_fig6c_top1000.py")
# -*- coding: utf-8 -*-
"""p0_8c_fig6c_top1000.py - Fig6C v32: gene-level two-channel landing heatmap of the MAGMA top1000 (= the Fig 6A/B scoring set)

The author noted that v31 went off domain: A/B use the MS GWAS MAGMA top1000 gene set, while 38/331 are FigS6A/B/E *cis*-window
locus calls - a different domain. Hence the main-figure C domain must match A/B = top1000.

Statistics domain (no circularity: scDRS scores are never touched, so no self-inclusion)
  rows = the 223 top-1000 genes measurable in this dataset (875 mapped - 652 near-zero, meta.json convention)
  display = the tested quantity: Δ[g] = 0.2810·L[MOL3/6] + 0.7190·L[MOL6] − 0, L[g,s] = mean(g,s) − mean(g,COP/NFOL)
  test    = set-mean Δ vs a detection-matched non-MS background (decile-stratified permutation, 10,000 draws) -> p08c_dual_setlevel_matched.csv
"""
import os, sys
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap
import fitz

CM = 1 / 2.54
HERE = os.path.dirname(os.path.abspath(__file__))
# Default = three levels above the repo GitHub/figS6/assembly (the analysis-tree root); override with DSCI_WORK_ROOT.
WORK = os.environ.get("DSCI_WORK_ROOT", os.path.abspath(os.path.join(HERE, "..", "..", "..")))
BASE = WORK + "/figures/routeA_out/P0_8C"
# In the repo the notebook points P08C_TABLES/S6_OUT at tables_v16 and fig6_out; the defaults keep the analysis-tree paths.
TAB = os.environ.get("P08C_TABLES", BASE + "/tables")
PAN = os.environ.get("S6_OUT", BASE + "/panels"); os.makedirs(PAN, exist_ok=True)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from arial_style import use_arial, use_two_tier
import fitz as _fz          # column labels need measured text widths; fitz is only imported for real in the legend section, so alias it here


def tw(t, fs=6.0):
    return _fz.get_text_length(t, fontname="helv", fontsize=fs) * 2.54 / 72.0
use_arial(); use_two_tier()

SUB = ["COP/NFOL", "MFOL", "MOL2", "MOL3", "MOL3/6", "MOL6"]
W = np.array([203, 490, 1016, 688, 1006, 2574], float)
fA = W[4] / (W[4] + W[5])
CONTRAST = os.environ.get("PANEL_CONTRAST", "pool")   # mol6 = the main-figure convention: Δ = MOL6 − COP/NFOL
JC = 5 if CONTRAST == "mol6" else None                 # None = cell-count weighting of the two enriched columns (old convention)
HAS_D = os.environ.get("PANEL_DCOLUMN", "1") == "1"    # the rightmost Δ column; the author decided to drop it once the single-column convention was set
SHOW_NAMES = os.environ.get("PANEL_NAMES", "1") == "1" # D (the null panel) stays unnamed
NSUB = 6
DETECT = 10.0
# ---- z display mode (author 2026-11-17, for the supplement) ----
# In the old Δ-vs-COP display the COP/NFOL column was identically 0 (the reference subtracted from itself) - a cell with no information;
# after per-gene standardization across the six subtypes the reference column itself enters the comparison (risk genes lower in COP -> blue),
# the six columns share one scale, so the RNA/ATAC blocks share ONE colour bar and D/E can share one legend.
ZMODE = os.environ.get("PANEL_ZMODE", "0") == "1"
DRAW_LEG = os.environ.get("PANEL_LEGEND", "1") == "1"
ZCLIP = float(os.environ.get("PANEL_ZCLIP", 2.0))
SC = {"rna": (-1.2, 1.2), "atac": (-0.35, 0.35)}      # different units: RNA log2(CP10k+1), ATAC gene score
GOLD, BLACK = "#b8860b", "#000000"
RNAH, ATAC_H = os.path.join(os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')), 'oligo/3k_dsci_oligo_subset.h5ad'), \
               os.path.join(os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')), 'oligo/fig6_rerun_20261019/dualomics_probe/genescore_oligo_5977.h5ad')

# ---------------------------------------------------------------- cell-level 223-gene sub-matrix (cached)
G = pd.read_csv(TAB + "/p08c_genelevel_all6_pool.csv")[["gene_human", "gene", "mean_pct"]]
cache = TAB + "/p08c_dual_celllevel223.npz"
if os.path.exists(cache):
    z = np.load(cache, allow_pickle=True)
    XG, AG, key = z["XG"], z["AG"], z["key"]
    print("using cache p08c_dual_celllevel223.npz", flush=True)
else:
    def grab(path, tag):
        a = ad.read_h5ad(path, backed="r")
        vn = np.array(list(map(str, a.var_names)))
        ci = np.array([int(np.flatnonzero(vn == g)[0]) if (vn == g).any() else -1 for g in G.gene])
        have = np.flatnonzero(ci >= 0)
        key = pd.Series(a.obs["cell_type"].astype(str)).str.upper().values   # keep '/', matching SUB
        acc = []
        for i in range(0, a.shape[0], 1024):
            X = a.X[i:i + 1024][:, ci[have]]
            X = X.toarray() if sp.issparse(X) else np.asarray(X)
            acc.append(X.astype(np.float32))
        full = np.full((a.shape[0], len(ci)), np.nan, np.float32); full[:, have] = np.vstack(acc)
        print("%s built %dx%d (ATAC missing %d genes)" % (tag, a.shape[0], len(ci), (ci < 0).sum()), flush=True)
        return full, key
    XG, key = grab(RNAH, "RNA")
    AG, _ = grab(ATAC_H, "ATAC")
    np.savez_compressed(cache, XG=XG, AG=AG, key=key, genes=G.gene.values)

rows = {s: np.flatnonzero(key == s) for s in SUB}


def summarize(M):
    with np.errstate(invalid="ignore"):
        Mn = np.column_stack([np.nanmean(M[rows[s]], 0) for s in SUB])
        Pc = np.column_stack([(np.nan_to_num(M[rows[s]]) > 0).mean(0) * 100 for s in SUB])
    return Mn, Pc


rng = np.random.default_rng(20261116)
res = {}
for tag, M in (("rna", XG), ("atac", AG)):
    Mn, Pc = summarize(M)
    L = Mn - Mn[:, :1]
    d = L[:, JC] if JC is not None else fA * L[:, 4] + (1 - fA) * L[:, 5]
    bs = []
    for _ in range(500):
        idx = {s: rng.choice(rows[s], len(rows[s]), replace=True) for s in SUB}
        with np.errstate(invalid="ignore"):
            p = np.column_stack([np.nanmean(M[idx[s]], 0) for s in SUB])
        bs.append((p[:, JC] - p[:, 0]) if JC is not None else
                  fA * (p[:, 4] - p[:, 0]) + (1 - fA) * (p[:, 5] - p[:, 0]))
    se = np.nanstd(np.array(bs), 0)
    res[tag] = dict(L=L, d=d, pct=Pc, se=se, ok=np.abs(d) > 2 * se)
    q = np.nanquantile(np.abs(L), [.5, .9, .95, 1.0])
    print("%s: Δ>0 %d/%d | sign-resolvable %d | |Δ| p50/p90 %.3f/%.3f | <10%%-detected cells %d/%d | landing |L| p50/90/95/max %.3f/%.3f/%.3f/%.3f"
          % (tag, (d > 0).sum(), d.size, res[tag]["ok"].sum(), np.nanmedian(np.abs(d)),
             np.nanquantile(np.abs(d), .9), (Pc < DETECT).sum(), Pc.size, *q), flush=True)

order = np.argsort(-res["rna"]["d"])
G = G.iloc[order].reset_index(drop=True)
for t in res:
    for k in ("L", "d", "pct", "se", "ok"):
        res[t][k] = res[t][k][order] if res[t][k].ndim == 2 else res[t][k][order]
NR = len(G)
rho = pd.DataFrame({"r": res["rna"]["d"], "a": res["atac"]["d"]}).corr(method="spearman").iloc[0, 1]
print("RNA x ATAC Δ Spearman = %.3f | sign-agreeing genes %d/%d" % (rho, ((np.sign(res["rna"]["d"]) == np.sign(res["atac"]["d"])) & res["rna"]["ok"] & res["atac"]["ok"]).sum(), NR))

# ---------------------------------------------------------------- layout (the v31 compliant geometry)
PW = float(os.environ.get("PANEL_W", 10.72)); PH = float(os.environ.get("PANEL_H", 9.60))
# In the narrow version (FigS6, 8.38 cm) the title must wrap: one 62-character line @8pt is ~9.2 cm > the 6.84 cm available,
# 'across' actually reached 8.43 cm and ran off the page (matplotlib text defaults to clip_on=False, so it was invisible and gates could not catch it).
WIDE = PW >= 10.0
TITLE = ("MS GWAS top-1000 gene set, per-gene landing across OL subtypes" if WIDE
         else "MS GWAS top-1000 gene set:\nper-gene landing across OL subtypes")
# Title externalized (author 2026-11-17): with PANEL_TITLE=0 no title is drawn on the face; the title text goes into the caption;
# the saved top space goes to the body, not to page height (row pitch is the only readability bottleneck of a 223-row heatmap).
TITLE_ON = os.environ.get("PANEL_TITLE", "1") == "1"
YA = PH - 0.35 if WIDE else PH - 0.20
YG = PH - 0.63 if WIDE else PH - 1.00
BODY_T = (PH - 0.70 if WIDE else PH - 1.08) if TITLE_ON else PH - 0.34
NARROW = PW < 10.0          # FigS6 narrow version: subtype labels hang vertically (45° would cut diagonally into the gene-name column)
BODY_B = float(os.environ.get("PANEL_BODY_B", (BODY_T - NR * 0.030) if not NARROW else 2.60))
RH = (BODY_T - BODY_B) / NR
# With the title externalized the body moves up, so the channel names "RNA"/"ATAC" must rise above the body - otherwise they sit on
# the first row's cells (verify_text_on_data would report DOT-UNDER-TEXT 'RNA').
if not TITLE_ON:
    # With va="baseline" the box BOTTOM edge sits ~0.045 cm below the baseline (the PDF emits boxes by the font's ascender/descender,
    # taller than matplotlib's extent, see SKILL §17.2); +0.03 would be pressed by the first row; top edge = BODY_T+0.25,
    # so the body needs at least 0.34 cm headroom to the page edge, else show_pdf_page trims at the CropBox (§17.6).
    YG = BODY_T + 0.06
LBLW, x_lbl, x_body = 1.14, 0.30, 1.54
CW = float(os.environ.get("PANEL_CW", 0.70))
# Baseline column width CW0: only the identically-zero COP/NFOL was narrowed on the raw scale; under z it carries information,
# and with the Δ column gone the six columns are equal, so it defaults to the same width as the rest (author 2026-11-17).
CW0 = float(os.environ.get("PANEL_CW0", CW if (ZMODE and not HAS_D) else 0.26))
NC = NSUB + (1 if HAS_D else 0)
XOFF = [0.0] + [CW0 + i * CW for i in range(NC - 1)]
BW = XOFF[-1] + CW
x_blk = [x_body, x_body + BW]
LEG_Y = 0.52
# With its own two-line legend the body needed >= 1.9 at the bottom; once the legend moved to the shared band, only the vertical subtype labels
# (longest "COP/NFOL" @6pt ~ 0.93 cm + 0.12 spacing) need to hang, so 1.20 suffices; the saved height all goes to the 223 rows.
assert BODY_B > (1.9 if DRAW_LEG else 1.20), BODY_B
print("body %.2f-%.2f (RH %.4f) | right edge %.2f" % (BODY_B, BODY_T, RH, x_blk[1] + BW))


def pick_labels(gap_cm=0.26, minpct=30.0, cap=14):
    need = max(1, int(np.ceil(gap_cm / RH)))
    cand = np.flatnonzero((res["rna"]["pct"].max(1) >= minpct) & res["rna"]["ok"])
    keep = []
    for i in sorted(cand, key=lambda x: -abs(res["rna"]["d"][x])):
        if all(abs(i - j) >= need for j in keep):
            keep.append(i)
        if len(keep) >= cap:
            break
    return sorted(keep), need


lab_rows, need = pick_labels()
if SHOW_NAMES:
    print("naming %d (spacing >= %d rows): %s" % (len(lab_rows), need, ", ".join(G.gene_human.iloc[lab_rows])))
else:
    print("PANEL_NAMES=0: this panel is unnamed (null panel); %d candidates recorded but not drawn: %s"
          % (len(lab_rows), ", ".join(G.gene_human.iloc[lab_rows])))

def rowz_disp(Ld):
    """Per gene: center on the six-subtype mean, then standardize; the denominator adds a shrinkage term s0 (= the median of per-gene sd).
    Without shrinkage, "nearly flat across the six columns" genes divide by a tiny sd and amplify noise into arbitrary strong colour - the classic z-heatmap trap.
    With shrinkage, flat genes sit at z ~ 0 (grey-white) and only structured genes open up into red/blue.
    The Δ column applies the SAME cell weights to the display → Δz = 0.2810·z[MOL3/6] + 0.7190·z[MOL6],
    still the weighted average of the two displayed columns, so a reader can verify it by eye against the colour bar; the invariant survives."""
    # A gene undetected everywhere is NaN for the whole row (greyed below by pct<DETECT), so centering/spread/shrink
    # must all use the nan* versions: one NaN would drag np.median of s0 to NaN and whiten the whole heatmap.
    mu = np.nanmean(Ld, 1, keepdims=True); sd = np.nanstd(Ld, 1, ddof=0, keepdims=True)
    s0 = float(np.nanmedian(sd))
    assert np.isfinite(s0) and s0 > 0, ("shrinkage s0 not finite", s0)
    Z = (Ld - mu) / np.sqrt(sd ** 2 + s0 ** 2)
    fz = W / W.sum()
    return Z, (Z[:, JC] if JC is not None else fz[4] * Z[:, 4] + fz[5] * Z[:, 5]), s0


if ZMODE:
    for _t in ("rna", "atac"):
        _Z, _DZ, _s0 = rowz_disp(res[_t]["L"])
        res[_t]["L"], res[_t]["d"] = _Z, _DZ
        print("[zmode] %s shrinkage s0 = %.4g | display switched to per-gene across-subtype z" % (_t, _s0), flush=True)
    SC = {"rna": (-ZCLIP, ZCLIP), "atac": (-ZCLIP, ZCLIP)}

fig = plt.figure(figsize=(PW / 2.54, PH / 2.54))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, PW); ax.set_ylim(0, PH); ax.axis("off")
CMAP = LinearSegmentedColormap.from_list("by", ["#2166ac", "#f7f7f7", "#b2182b"])
if TITLE_ON:
    ax.text(x_body, YA, TITLE, fontsize=8.0, ha="left",
            va="baseline" if WIDE else "top", linespacing=1.3, color=BLACK)


def cell_y(i):
    return BODY_T - (i + 1) * RH


for nm, x0 in (("RNA", x_blk[0]), ("ATAC", x_blk[1])):
    LO, HI = SC[nm.lower()]
    R = res[nm.lower()]; L, d, pct, ok = R["L"], R["d"], R["pct"], R["ok"]
    for i in range(NR):
        y, h = cell_y(i), RH - 0.003
        for j in range(NSUB):
            v, wx, ww = L[i, j], x0 + XOFF[j], (CW0 if j == 0 else CW)
            if pct[i, j] < DETECT:
                ax.add_patch(Rectangle((wx, y), ww, h, fc="#d9d9d9", ec="none"))
            else:
                t = max(min(v, HI), LO)
                ax.add_patch(Rectangle((wx, y), ww, h, fc=CMAP((t - LO) / (HI - LO)), ec="none"))
                if abs(v) > HI:   # the reference column (j=0) is marked too: under z it is no longer identically 0 and all off-scale cells sit there (the old j>0 was a leftover that swallowed every mark)
                    ax.plot(wx + (ww - 0.05 if v > 0 else 0.05), y + h / 2, marker=">" if v > 0 else "<",
                            ms=1.2, color="#ffffff", mew=0.3, zorder=6)
        if not HAS_D:
            continue
        wx = x0 + XOFF[NC - 1]
        if ok[i]:
            t = max(min(d[i], HI), LO)
            ax.add_patch(Rectangle((wx, y), CW, h, fc=CMAP((t - LO) / (HI - LO)), ec="none"))
            if abs(d[i]) > HI:
                ax.plot(wx + (CW - 0.05 if d[i] > 0 else 0.05), y + h / 2, marker=">" if d[i] > 0 else "<",
                        ms=1.2, color="#ffffff", mew=0.3, zorder=6)
        else:
            ax.add_patch(Rectangle((wx, y), CW, h, fc="#d9d9d9", ec="none"))
    _n = int(((np.abs(L) > HI) & (pct >= DETECT)).sum())
    print("  %s block off-scale triangles %d (of which reference column COP/NFOL %d) | |z|>%.1f and not greyed"
          % (nm, _n, int(((np.abs(L[:, 0]) > HI) & (pct[:, 0] >= DETECT)).sum()), HI), flush=True)
    ax.add_patch(Rectangle((x0, BODY_B), BW, BODY_T - BODY_B, fc="none", ec="#333333", lw=0.4))
    if HAS_D:
        ax.add_patch(Rectangle((x0 + XOFF[NC - 1], BODY_B), CW, BODY_T - BODY_B, fc="none", ec=BLACK, lw=0.7))
    ax.text(x0 + BW / 2, YG, nm, fontsize=6.0, ha="center", va="baseline", color=BLACK)
    for j, s in enumerate(SUB + (["\u0394"] if HAS_D else [])):
        ww = CW0 if j == 0 else CW
        if NARROW:
            # The MEASURED semantics of rotation=90 + ha=right + va=center: the anchor lands at the text's vertical centre,
            # horizontal right edge. To hang it top-aligned under the body, drop the anchor by L/2 per label's measured length.
            ax.text(x0 + XOFF[j] + ww / 2 + 0.115, BODY_B - 0.12 - tw(s) / 2.0, s,
                    fontsize=6.0, rotation=90, ha="right", va="center", color=BLACK)
        else:
            ax.text(x0 + XOFF[j] + ww / 2, BODY_B - 0.07, s, fontsize=6.0, rotation=45,
                    ha="right", va="top", color=BLACK)
    for k in ((JC,) if JC is not None else (4, 5)):
        ax.add_patch(Rectangle((x0 + XOFF[k] + 0.06, BODY_B - 0.055), CW - 0.12, 0.035, fc=GOLD, ec="none"))
    if not HAS_D:   # with no Δ column, "which is the numerator and which the denominator" rests entirely on these two identity bars
        ax.add_patch(Rectangle((x0 + XOFF[0] + 0.06, BODY_B - 0.055), CW0 - 0.12, 0.035, fc=BLACK, ec="none"))

for i in (lab_rows if SHOW_NAMES else []):
    ax.text(x_lbl + LBLW - 0.04, cell_y(i) + (RH - 0.003) / 2, G.gene_human[i], fontsize=6.0,
            ha="right", va="center", color=BLACK)

# ---------------------------------------------------------------- legend
def tw(t, fs=6.0):
    return fitz.get_text_length(t, fontname="helv", fontsize=fs) * 2.54 / 72.0


def legend(x0, x1, y, items, gap=0.42):
    MKW = {"grad": 1.45, "sw": 0.26, "car": 0.30, "bar": 0.42}
    wid = [MKW[k] + (0.0 if k == "grad" else 0.06 + tw(t)) for k, t in items]
    tot = sum(wid) + gap * (len(items) - 1)
    if x0 + tot > x1:
        raise SystemExit("legend too wide %.2f > %.2f" % (x0 + tot, x1))
    x = x0 + (x1 - x0 - tot) / 2.0
    for (kind, t), w in zip(items, wid):
        if kind == "grad":
            ax.imshow(np.linspace(HI, LO, 64)[None, :], extent=(x, x + MKW[kind], y - 0.075, y + 0.075),
                      cmap=CMAP, aspect="auto", zorder=3)
            ax.add_patch(Rectangle((x, y - 0.075), MKW[kind], 0.15, fc="none", ec="#333333", lw=0.3))
            lo_, hi_ = SC[{"RNA log2": "rna", "ATAC score": "atac", "z": "rna"}[t]]
            for px, lab in ((x, "%+.2f" % hi_ if hi_ < 1 else "%+.1f" % hi_),
                            (x + MKW[kind] / 2, "0"),
                            (x + MKW[kind], ("\u2212%.2f" % abs(lo_)) if abs(lo_) < 1 else ("\u2212%.1f" % abs(lo_)))):
                ax.text(px, y - 0.115, lab, fontsize=6.0, ha="center", va="top", color=BLACK)
            ax.text(x + MKW[kind] / 2, y + 0.115, t, fontsize=6.0, ha="center", va="bottom", color=BLACK)
        elif kind == "sw":
            ax.add_patch(Rectangle((x, y - 0.075), MKW[kind], 0.15, fc="#d9d9d9", ec="#8c8c8c", lw=0.3))
            ax.text(x + MKW[kind] + 0.06, y, t, fontsize=6.0, ha="left", va="center", color=BLACK)
        elif kind == "car":
            ax.plot(x + 0.07, y, marker="<", ms=2.2, color="#2166ac", mew=0.3)
            ax.plot(x + 0.23, y, marker=">", ms=2.2, color="#b2182b", mew=0.3)
            ax.text(x + MKW[kind] + 0.06, y, t, fontsize=6.0, ha="left", va="center", color=BLACK)
        else:
            ax.add_patch(Rectangle((x, y - 0.055), MKW[kind], 0.075, fc=GOLD, ec="none"))
            ax.text(x + MKW[kind] + 0.06, y, t, fontsize=6.0, ha="left", va="center", color=BLACK)
        x += w + gap
    return x - gap


if not DRAW_LEG:
    print("legend handed to the shared band (PANEL_LEGEND=0)", flush=True)
    xe = x_body
    _it = []
else:
  _it = ([("grad", "z"), ("sw", "not detected"), ("car", "off scale"), ("bar", "tested")]
         if ZMODE else
         [("grad", "RNA log2"), ("grad", "ATAC score"), ("sw", "not detected"),
          ("car", "off scale"), ("bar", "tested")])
if _it and PW < 10.0:      # narrow two-line legend (as above)
    xe = legend(x_body, x_blk[1] + BW, LEG_Y, _it[2:], gap=0.90)
    # the two gradient bars' end ticks centre on the bar ends; too small a gap reads as "−1.2+0.35" (gates silent, eyes misread) -> separate them
    xe = legend(x_body, x_blk[1] + BW, LEG_Y + 0.44, _it[:2], gap=1.10)
elif _it:
    xe = legend(x_body, x_blk[1] + BW, LEG_Y, _it)
if _it:
    print("legend %.2f-%.2f" % (x_body, xe))

out = PAN + "/fig6v32_C_top1000_landing%s.pdf" % os.environ.get("PANEL_TAG", "")
fig.savefig(out, dpi=600)
d = fitz.open(out); p = d[0]
Wp, Hp = p.rect.width * 2.54 / 72, p.rect.height * 2.54 / 72
ws = p.get_text("words")
worst = max((w[2] * 2.54 / 72 for w in ws), default=0); bot = max((w[3] * 2.54 / 72 for w in ws), default=0)
sp_ = [s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]]
sizes = sorted({round(s["size"], 2) for s in sp_})
grey = sorted({"%06x" % s["color"] for s in sp_ if s["color"] not in (0, 0xffffff)})
print("size %.2f x %.2f | text right %.2f (limit %.2f) | bottom %.2f | sizes %s | non-black text %s"
      % (Wp, Hp, worst, Wp - 0.20, bot, sizes, grey))
d.close()

using cache p08c_dual_celllevel223.npz
rna: Δ>0 54/223 | sign-resolvable 122 | |Δ| p50/p90 0.092/0.317 | <10%-detected cells 255/1338 | landing |L| p50/90/95/max 0.084/0.323/0.468/2.451


/tmp/ipykernel_63420/265704981.py:100: RuntimeWarning: Mean of empty slice
  Mn = np.column_stack([np.nanmean(M[rows[s]], 0) for s in SUB])
/tmp/ipykernel_63420/265704981.py:115: RuntimeWarning: Mean of empty slice
  p = np.column_stack([np.nanmean(M[idx[s]], 0) for s in SUB])


atac: Δ>0 77/223 | sign-resolvable 125 | |Δ| p50/p90 0.045/0.169 | <10%-detected cells 81/1338 | landing |L| p50/90/95/max 0.034/0.148/0.202/0.505
RNA x ATAC Δ Spearman = 0.650 | sign-agreeing genes 76/223
body 1.35-5.41 (RH 0.0182) | right edge 7.21
PANEL_NAMES=0: this panel is unnamed (null panel); 9 candidates recorded but not drawn: PRR5L, ST3GAL4, DPH6, UBAC2, UBE2D2, PCDHGA1, DIP2C, CLEC16A, FCHSD2
[zmode] rna shrinkage s0 = 0.07755 | display switched to per-gene across-subtype z
[zmode] atac shrinkage s0 = 0.02619 | display switched to per-gene across-subtype z


~/.local/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_63420/265704981.py:200: RuntimeWarning: Mean of empty slice
  mu = np.nanmean(Ld, 1, keepdims=True); sd = np.nanstd(Ld, 1, ddof=0, keepdims=True)


  RNA block off-scale triangles 3 (of which reference column COP/NFOL 3) | |z|>2.0 and not greyed
  ATAC block off-scale triangles 11 (of which reference column COP/NFOL 11) | |z|>2.0 and not greyed
legend handed to the shared band (PANEL_LEGEND=0)
size 7.45 x 5.75 | text right 7.09 (limit 7.25) | bottom 5.60 | sizes [6.0] | non-black text []


## 5. Panel E — the cis-window set lands on MOL6

38 genes whose platform gene lies inside a lead-SNP ±500 kb window, same scale, same contrast.
Here the tested result is positive in both channels: mean Δ = +0.145 (RNA) / +0.097 (ATAC)
against detection-breadth × signal-matched backgrounds, *z* = +2.87 / +3.07,
*P* = 0.0017 / 0.0007; the myelin/oligodendrocyte anchor set gives 0.0107 / 0.0080 and the same
panel's 293 non-window platform genes give 0.0127 / 0.0066 (leave-one-gene-out worst 0.0127
dropping PRR5L, 0.0038 dropping KCTD13).

**E is not a subset of D.**  Only 8 of the 38 window genes appear in the MS top-1000 at all, and
6 of those are measurable here, so the two panels share 6 rows. Two independent gene-level
definitions of the same GWAS, disagreeing — which is the finding the two panels are there to show.

The four named genes are the panel-A loci (PHLDB1, PHGDH, PLLP, BCAS1). Naming is independent of
the test: row order and the permutation both use Δ, and the names come from positional evidence
that is not on this panel's face. BCAS1 is the one that a reader may question: it is the
*progenitor*-end gene of the four (top subtype COP/NFOL, Δ_MOL6 = −1.71, ranked 38/38 within the
set, and −1.14 in the independent human mature-vs-precursor contrast), it is **not** in the MS
top-1000, so its negative Δ is not a counter-example to Fig 6A/B — it is the reason the manuscript
calls the BCAS1 coincidence a coincidence of gene identity.

In [9]:
# run recipe for this panel
os.environ["PANEL_TAG"] = "_s6v18E"
os.environ["PANEL_CONTRAST"] = "mol6"
os.environ["PANEL_ZMODE"] = "1"
os.environ["PANEL_DCOLUMN"] = "0"
os.environ["PANEL_NAMES"] = "1"        # E names its genes; the D cell leaves the shared env at 0, so this restores 1 explicitly
os.environ["PANEL_PRESPEC"] = "PHLDB1,PHGDH,BCAS1,PLLP"   # same env as figS6v16_run_all.sh
# ------------------------------------------------------------ assembly/p0_8c_fig6c_landing.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/p0_8c_fig6c_landing.py")
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
p0_8c_fig6c_landing.py - Fig6C v31: the "landing" heatmap of 38 *cis*-window risk genes x 6 OL subtypes (main-figure version)

v30 -> v31 compliance fixes (against skills/figure-house-spec/SKILL.md)
  1. §8.3/§2  all five face explanation lines deleted: reading-essential marks go into the swatch legend (§12); the rest (Δ definition, bootstrap,
     ARI, permutation P, detection-rate counts) all go into the caption (FIG6C_v31_CAPTION.md).
  2. §14.1    the 0.42 cm gap between the RNA/ATAC blocks was a direct FAIL, so the columns now touch end-to-end, separated by a thin line only;
     the freed width goes to the columns (0.55 -> 0.70 cm/column).
  3. §2        all face text is pure BLACK (the #333/#666 greys were violations); enriched subtypes no longer use gold text,
     replaced by a GOLD identity bar under the column (§14.1 permits a column-identity bar; it is a mark, not a text colour).
  4. §2        titles lose bold (face defaults to regular black text, no bold, no italics).
  5. the colour scale moved from a right-hand vertical bar to a horizontal gradient bar INSIDE the legend, eliminating overlap with the graphic; the end ticks centre exactly on the bar ends x (§14.11).

Display = the tested quantity, and the tested quantity is checkable by eye
  landing column L[g,s] = mean_log2(g,s) − mean_log2(g,COP/NFOL) (the COP/NFOL column is identically 0, narrowed as the explicit baseline)
  Δ column Δ[g]   = (1006·L[MOL3/6] + 2574·L[MOL6])/3580 − 0 = the weighted mean of the two enriched columns
"""
import os, sys
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap

CM = 1 / 2.54
HERE = os.path.dirname(os.path.abspath(__file__))
# Default = three levels above the repo GitHub/figS6/assembly (the analysis-tree root); override with DSCI_WORK_ROOT.
WORK = os.environ.get("DSCI_WORK_ROOT", os.path.abspath(os.path.join(HERE, "..", "..", "..")))
BASE = WORK + "/figures/routeA_out/P0_8C"
# In the repo the notebook points P08C_TABLES/S6_OUT at tables_v16 and fig6_out; the defaults keep the analysis-tree paths.
TAB = os.environ.get("P08C_TABLES", BASE + "/tables")
PAN = os.environ.get("S6_OUT", BASE + "/panels"); os.makedirs(PAN, exist_ok=True)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from arial_style import use_arial, use_two_tier
import fitz as _fz          # column labels need measured text widths; fitz is only imported for real in the legend section, so alias it here


def tw(t, fs=6.0):
    return _fz.get_text_length(t, fontname="helv", fontsize=fs) * 2.54 / 72.0
use_arial(); use_two_tier()

SUB = ["COP/NFOL", "MFOL", "MOL2", "MOL3", "MOL3/6", "MOL6"]
ENA, ENB = "MOL3/6", "MOL6"
# MOL3/6=1006 is the full RNA AnnData used by the C1 pre-check; the 991 on the Fig 6A axis is the subset with scDRS scores (15 cells fewer)
NCELL = dict(zip(SUB, [203, 490, 1016, 688, 1006, 2574]))
LO, HI = -1.2, 1.2
DETECT_MIN = 10.0
# z display mode: same convention as p0_8c_fig6c_top1000.py (see its comments); both blocks share one colour bar / legend.
ZMODE = os.environ.get("PANEL_ZMODE", "0") == "1"
DRAW_LEG = os.environ.get("PANEL_LEGEND", "1") == "1"
ZCLIP = float(os.environ.get("PANEL_ZCLIP", 2.0))
CONTRAST = os.environ.get("PANEL_CONTRAST", "pool")   # mol6 = the main-figure convention: Δ = MOL6 − COP/NFOL
JC = 5 if CONTRAST == "mol6" else None                 # None = cell-count weighting of the two enriched columns (old convention)
HAS_D = os.environ.get("PANEL_DCOLUMN", "1") == "1"    # the rightmost Δ column; the author decided to drop it once the single-column convention was set
SHOW_NAMES = os.environ.get("PANEL_NAMES", "1") == "1" # D (the null panel) stays unnamed
NSUB = 6
GOLD = "#b8860b"          # channel/subtype identity colour, marks only, never text
BLACK = "#000000"

# ---------------------------------------------------------------- data (single source: the cell-level cached npz)
S0 = pd.read_csv(TAB + "/p08c_c1_rna_atac_by_subtype.csv")[["gene", "mouse", "top_subtype"]]
zz = np.load(TAB + "/p08c_c2_celllevel38.npz", allow_pickle=True)
XG, AG, key, cg = zz["XG"], zz["AG"], zz["key"], [str(g) for g in zz["genes"]]
rows = {s: np.flatnonzero(key == s) for s in SUB}
col = np.array([cg.index(m) for m in S0.mouse])


def summarize(M):
    Mn = np.column_stack([M[rows[s]][:, col].mean(0) for s in SUB]).astype(float)
    Pc = np.column_stack([(M[rows[s]][:, col] > 0).mean(0) * 100 for s in SUB]).astype(float)
    return Mn, Pc


Mn_r, pct_r = summarize(XG)
Mn_a, pct_a = summarize(AG)
Lr, La = Mn_r - Mn_r[:, :1], Mn_a - Mn_a[:, :1]
wA, wB = NCELL[ENA], NCELL[ENB]
if JC is None:
    d_r = (wA * Mn_r[:, 4] + wB * Mn_r[:, 5]) / (wA + wB) - Mn_r[:, 0]
    d_a = (wA * Mn_a[:, 4] + wB * Mn_a[:, 5]) / (wA + wB) - Mn_a[:, 0]
    assert np.allclose(d_r, (wA * Lr[:, 4] + wB * Lr[:, 5]) / (wA + wB), atol=1e-9)  # Δ is a linear combination of the landing columns
else:            # main-figure convention: only compared with MOL6, so cell weights are no longer needed
    d_r, d_a = Lr[:, JC].copy(), La[:, JC].copy()

rng = np.random.default_rng(20261116); br, ba = [], []
for _ in range(500):
    idx = {s: rng.choice(rows[s], len(rows[s]), replace=True) for s in SUB}
    for M, acc in ((XG, br), (AG, ba)):
        p = np.column_stack([M[idx[s]][:, col].mean(0) for s in SUB])
        acc.append((p[:, JC] - p[:, 0]) if JC is not None else
                   (wA * p[:, 4] + wB * p[:, 5]) / (wA + wB) - p[:, 0])
SE_R, SE_A = np.array(br).std(0), np.array(ba).std(0)
res_r, res_a = np.abs(d_r) > 2 * SE_R, np.abs(d_a) > 2 * SE_A

comp = d_r / d_r.std() + d_a / d_a.std()
order = np.argsort(-comp)
S = S0.iloc[order].reset_index(drop=True)
Lr, La, d_r, d_a = Lr[order], La[order], d_r[order], d_a[order]
# [BUG FIX] comp must be reordered together with order. Previously only S/L/d/res/pct were reordered, so inside pick_labels
# `key=lambda x: -abs(comp[x])` looked up the UNreordered array by row x - i.e. ranked priority by another gene's effect size.
# Consequence: pool, row order and name-to-row mapping were ALL correct (no face mislabel), but "who deserves the name" was wrong -
# 4 of the 10 names were not the ones the rule should have picked. Neither the three gates nor visual review can catch this.
comp = comp[order]
res_r, res_a, pct_r, pct_a = res_r[order], res_a[order], pct_r[order], pct_a[order]
print("sign-resolvable RNA %d/38 ATAC %d/38 | <10%%-detected landing cells RNA %d ATAC %d (of 228)"
      % (res_r.sum(), res_a.sum(), (pct_r < DETECT_MIN).sum(), (pct_a < DETECT_MIN).sum()))

# ---------------------------------------------------------------- layout
# Panel size is env-overridable (the FigS6 three-row version needs 8.38 wide); unset = the delivered main-figure size as-is.
PW = float(os.environ.get("PANEL_W", 10.72)); PH = float(os.environ.get("PANEL_H", 5.50))
NR = 38
NC = NSUB + (1 if HAS_D else 0)
LBLW = 1.14
x_lbl, x_body = 0.30, 1.54
TITLE_ON = os.environ.get("PANEL_TITLE", "1") == "1"   # as in D: title goes to the caption, the saved height to the body
YA, YG = PH - 0.27, PH - 0.55
BODY_T = (PH - 0.62) if TITLE_ON else PH - 0.34
NARROW = PW < 10.0          # FigS6 narrow version: subtype labels hang vertically (45° would cut diagonally into the gene-name column)
BODY_B = float(os.environ.get("PANEL_BODY_B", 2.05 if PW >= 10.0 else 2.60))
CW0 = float(os.environ.get("PANEL_CW0",
            float(os.environ.get("PANEL_CW", 0.70)) if (ZMODE and not HAS_D) else 0.26))
CW = float(os.environ.get("PANEL_CW", 0.70))
XOFF = [0.0] + [CW0 + i * CW for i in range(NC - 1)]
BW = XOFF[-1] + CW
x_blk = [x_body, x_body + BW]                          # the two blocks touch end-to-end, no gap (§14.1)
RH = (BODY_T - BODY_B) / NR
# With the title externalized the body moves up, so the channel names "RNA"/"ATAC" must rise above the body - otherwise they sit on
# the first row's cells (verify_text_on_data would report DOT-UNDER-TEXT 'RNA').
if not TITLE_ON:
    # With va="baseline" the box BOTTOM edge sits ~0.045 cm below the baseline (the PDF emits boxes by the font's ascender/descender,
    # taller than matplotlib's extent, see SKILL §17.2); +0.03 would be pressed by the first row; top edge = BODY_T+0.25,
    # so the body needs at least 0.34 cm headroom to the page edge, else show_pdf_page trims at the CropBox (§17.6).
    YG = BODY_T + 0.06
LEG_Y = 0.52
assert x_blk[1] + BW <= PW - 0.20, x_blk[1] + BW


def pick_labels(gap_cm=0.26):
    """Naming rule for rows (stated in the caption, never hand-picked): max detection >= 30% and RNA sign-resolvable (|Δ|>2SE),
    then greedy naming by descending |composite Δ|, named rows at least `need` rows apart (so 6pt labels never collide)."""
    need = max(1, int(np.ceil(gap_cm / RH)))
    cand = np.flatnonzero((pct_r.max(1) >= 30) & res_r)
    keep = []
    for i in sorted(cand, key=lambda x: -abs(comp[x])):
        if all(abs(i - j) >= need for j in keep):
            keep.append(i)
    return sorted(keep)


PRESPEC = [g.strip() for g in os.environ.get("PANEL_PRESPEC", "").split(",") if g.strip()]
if PRESPEC:
    """The prespecified list wins (author 2026-11-17). The largest-|Δ| genes are picked BY THE TESTED QUANTITY ITSELF,
    using them as evidence is circular; instead, only the genes that panel S6A resolved independently by locus are named -
    the names' justification lives OFF the face (GWAS lead SNP ±500 kb), orthogonal to the panel statistics.
    The spacing rule yields to the prespecified list: colliding labels are pushed up and pointed back to their row with a short leader."""
    have = set(S.gene)
    miss = [g for g in PRESPEC if g not in have]
    assert not miss, ("prespecified gene not among the panel rows", miss)
    lab_rows = sorted(int(np.flatnonzero(S.gene.values == g)[0]) for g in PRESPEC)
else:
    lab_rows = pick_labels()
print("naming %d:" % len(lab_rows), ", ".join(S.gene.iloc[lab_rows]))

fig = plt.figure(figsize=(PW / 2.54, PH / 2.54))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, PW); ax.set_ylim(0, PH); ax.axis("off")
CMAP = LinearSegmentedColormap.from_list("by", ["#2166ac", "#f7f7f7", "#b2182b"])
if TITLE_ON:
    ax.text(x_body, YA, "Risk-set landing across OL subtypes", fontsize=8.0, ha="left",
            va="baseline", color=BLACK)


def cell_y(i):
    return BODY_T - (i + 1) * RH


def rowz_disp(Ld):
    """Same function as the top1000 panel: the denominator adds a shrinkage term s0; Δ applies the same cell weights to the display.
    Row order still follows the delivered "composite Δ" (the raw quantity) - the caption states that rule; it does not change with the display."""
    # A gene undetected everywhere is NaN for the whole row (greyed below by pct<DETECT_MIN), so centering/spread/shrink
    # must all use the nan* versions: one NaN would drag np.median of s0 to NaN and whiten the whole heatmap.
    mu = np.nanmean(Ld, 1, keepdims=True); sd = np.nanstd(Ld, 1, ddof=0, keepdims=True)
    s0 = float(np.nanmedian(sd))
    assert np.isfinite(s0) and s0 > 0, ("shrinkage s0 not finite", s0)
    Z = (Ld - mu) / np.sqrt(sd ** 2 + s0 ** 2)
    fz = np.array([NCELL[s] for s in SUB], float); fz = fz / fz.sum()
    return Z, (Z[:, JC] if JC is not None else fz[4] * Z[:, 4] + fz[5] * Z[:, 5]), s0


if ZMODE:
    Lr, d_r, _s0r = rowz_disp(Lr)
    La, d_a, _s0a = rowz_disp(La)
    LO, HI = -ZCLIP, ZCLIP
    print("[zmode] RNA s0 = %.4g | ATAC s0 = %.4g" % (_s0r, _s0a), flush=True)

_placed = []
LBL_H = 0.22   # the real 6pt gene-name glyph height (ascender+descender): the push-up collision step
for b, (Mat, x0, dlt, res, pct, nm) in enumerate(((Lr, x_blk[0], d_r, res_r, pct_r, "RNA"),
                                                  (La, x_blk[1], d_a, res_a, pct_a, "ATAC"))):
    for i in range(NR):
        y, h = cell_y(i), RH - 0.004
        for j in range(NSUB):
            v, wx, ww = Mat[i, j], x0 + XOFF[j], (CW0 if j == 0 else CW)
            if pct[i, j] < DETECT_MIN:
                ax.add_patch(Rectangle((wx, y), ww, h, fc="#d9d9d9", ec="none"))
            else:
                t = max(min(v, HI), LO)
                ax.add_patch(Rectangle((wx, y), ww, h, fc=CMAP((t - LO) / (HI - LO)), ec="none"))
                if abs(v) > HI:   # the reference column (j=0) is marked too: under z it is no longer identically 0 and all off-scale cells sit there (the old j>0 was a leftover that swallowed every mark)
                    ax.plot(wx + (ww - 0.07 if v > 0 else 0.07), y + h / 2, marker=">" if v > 0 else "<",
                            ms=1.5, color="#ffffff", mew=0.4, zorder=6)
        if not HAS_D:
            continue
        wx = x0 + XOFF[NC - 1]
        if res[i]:
            t = max(min(dlt[i], HI), LO)
            ax.add_patch(Rectangle((wx, y), CW, h, fc=CMAP((t - LO) / (HI - LO)), ec="none"))
            if abs(dlt[i]) > HI:
                ax.plot(wx + (CW - 0.07 if dlt[i] > 0 else 0.07), y + h / 2,
                        marker=">" if dlt[i] > 0 else "<", ms=1.5, color="#ffffff", mew=0.4, zorder=6)
        else:
            ax.add_patch(Rectangle((wx, y), CW, h, fc="#d9d9d9", ec="none"))
    _n = int(((np.abs(Mat) > HI) & (pct >= DETECT_MIN)).sum())      # this panel's display is called Mat, not L
    print("  %s block off-scale triangles %d (of which reference column COP/NFOL %d) | |z|>%.1f and not greyed"
          % (nm, _n, int(((np.abs(Mat[:, 0]) > HI) & (pct[:, 0] >= DETECT_MIN)).sum()), HI), flush=True)
    ax.add_patch(Rectangle((x0, BODY_B), BW, BODY_T - BODY_B, fc="none", ec="#333333", lw=0.4))
    if HAS_D:
        ax.add_patch(Rectangle((x0 + XOFF[NC - 1], BODY_B), CW, BODY_T - BODY_B, fc="none", ec=BLACK, lw=0.7))
    ax.text(x0 + BW / 2, YG, nm, fontsize=6.0, ha="center", va="baseline", color=BLACK)
    for j, s in enumerate(SUB + (["\u0394"] if HAS_D else [])):
        ww = CW0 if j == 0 else CW
        if NARROW:
            # The MEASURED semantics of rotation=90 + ha=right + va=center: the anchor lands at the text's vertical centre,
            # horizontal right edge. To hang it top-aligned under the body, drop the anchor by L/2 per label's measured length.
            ax.text(x0 + XOFF[j] + ww / 2 + 0.115, BODY_B - 0.12 - tw(s) / 2.0, s,
                    fontsize=6.0, rotation=90, ha="right", va="center", color=BLACK)
        else:
            ax.text(x0 + XOFF[j] + ww / 2, BODY_B - 0.07, s, fontsize=6.0, rotation=45,
                    ha="right", va="top", color=BLACK)
    # the gold identity bar under the enriched subtype (a mark, not a text colour)
    for k in ((JC,) if JC is not None else (4, 5)):
        ax.add_patch(Rectangle((x0 + XOFF[k] + 0.06, BODY_B - 0.055), CW - 0.12, 0.035, fc=GOLD, ec="none"))
    if not HAS_D:   # with no Δ column, "which is the numerator and which the denominator" rests entirely on these two identity bars
        ax.add_patch(Rectangle((x0 + XOFF[0] + 0.06, BODY_B - 0.055), CW0 - 0.12, 0.035, fc=BLACK, ec="none"))

for i in sorted(lab_rows, reverse=True) if SHOW_NAMES else []:   # bottom-up
    yc = cell_y(i) + (RH - 0.004) / 2
    y = max(yc, _placed[-1] + LBL_H) if _placed else yc
    _placed.append(y)
    ax.text(x_lbl + LBLW - 0.04, y, S.gene[i], fontsize=6.0, ha="right", va="center", color=BLACK)
    if abs(y - yc) > 0.015:                                  # leader: name's right end -> its own row's left edge
        ax.plot([x_lbl + LBLW - 0.01, x_body - 0.05], [y, yc], lw=0.3, color="#333333", zorder=5)

# ---------------------------------------------------------------- swatch legend (§12: marks and their text share one coordinate system)
import fitz
def tw(t, fs=6.0):
    return fitz.get_text_length(t, fontname="helv", fontsize=fs) * 2.54 / 72.0


def legend(x0, x1, y, items, gap=0.42):
    """items = [(kind, text)]. With kind='grad' the text sits ABOVE the gradient bar (not consuming horizontal width),
    the end-tick text centres exactly on the bar-end x (§14.11). Too wide fails hard; never solved by shrinking type."""
    MKW = {"grad": 1.45, "sw": 0.26, "car": 0.30, "bar": 0.42}
    wid = [MKW[k] + (0.0 if k == "grad" else 0.06 + tw(t)) for k, t in items]
    tot = sum(wid) + gap * (len(items) - 1)
    if x0 + tot > x1:
        raise SystemExit("legend too wide %.2f > %.2f - please shorten the wording" % (x0 + tot, x1))
    x = x0 + (x1 - x0 - tot) / 2.0
    for (kind, t), w in zip(items, wid):
        if kind == "grad":
            ax.imshow(np.linspace(HI, LO, 64)[None, :], extent=(x, x + MKW[kind], y - 0.075, y + 0.075),
                      cmap=CMAP, aspect="auto", zorder=3)
            ax.add_patch(Rectangle((x, y - 0.075), MKW[kind], 0.15, fc="none", ec="#333333", lw=0.3))
            for px, lab in ((x, "%+.1f" % HI), (x + MKW[kind] / 2, "0"),
                            (x + MKW[kind], "\u2212%.1f" % abs(LO))):
                ax.text(px, y - 0.115, lab, fontsize=6.0, ha="center", va="top", color=BLACK)
            ax.text(x + MKW[kind] / 2, y + 0.115, t, fontsize=6.0, ha="center", va="bottom", color=BLACK)
        elif kind == "sw":
            ax.add_patch(Rectangle((x, y - 0.075), MKW[kind], 0.15, fc="#d9d9d9", ec="#8c8c8c", lw=0.3))
            ax.text(x + MKW[kind] + 0.06, y, t, fontsize=6.0, ha="left", va="center", color=BLACK)
        elif kind == "car":
            ax.plot(x + 0.07, y, marker="<", ms=2.2, color="#2166ac", mew=0.3)
            ax.plot(x + 0.23, y, marker=">", ms=2.2, color="#b2182b", mew=0.3)
            ax.text(x + MKW[kind] + 0.06, y, t, fontsize=6.0, ha="left", va="center", color=BLACK)
        else:
            ax.add_patch(Rectangle((x, y - 0.055), MKW[kind], 0.075, fc=GOLD, ec="none"))
            ax.text(x + MKW[kind] + 0.06, y, t, fontsize=6.0, ha="left", va="center", color=BLACK)
        x += w + gap
    return x - gap


if DRAW_LEG:
  _it = [("grad", "z" if ZMODE else "log2 vs COP/NFOL"), ("sw", "not detected"),
         ("car", "off scale"), ("bar", "enriched subtypes")]
  if PW < 10.0:      # narrow version (FigS6 8.38 cm): the legend wraps to two rows; wording kept, type not shrunk (spec §3.3 breathes first)
      # when the two gradient bars sit side by side, the end ticks "−1.2" and "+0.35" read as one string (gates silent, eyes misread) -> widen the gap
      xe = legend(x_body, x_blk[1] + BW, LEG_Y, _it[2:], gap=0.90)
      # each gradient bar's end ticks centre on its bar end: if the gap is smaller than the sum of the half-widths of the two labels it reads "−1.2+0.35" (gates silent, eyes misread)
      xe = legend(x_body, x_blk[1] + BW, LEG_Y + 0.44, _it[:2], gap=1.10)
  else:
      xe = legend(x_body, x_blk[1] + BW, LEG_Y, _it)
  print("legend occupies %.2f-%.2f (available %.2f-%.2f)" % (x_body, xe, x_body, x_blk[1] + BW))
else:
    xe = x_body
    print("legend handed to the shared band (PANEL_LEGEND=0)", flush=True)

out = PAN + "/fig6v31_C_landing%s.pdf" % os.environ.get("PANEL_TAG", "")
fig.savefig(out, dpi=600)
C = 2.54 / 72
d = fitz.open(out); p = d[0]
W, H = p.rect.width * C, p.rect.height * C
ws = p.get_text("words")
worst = max((w[2] * C for w in ws), default=0); bot = max((w[3] * C for w in ws), default=0)
sp = [s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]]
grey = sorted({"%06x" % s["color"] for s in sp if s["color"] not in (0, 0xffffff)})
print("size %.2f x %.2f | text right %.2f (limit %.2f) | bottom %.2f | non-black text %s" % (W, H, worst, W - 0.20, bot, grey))
d.close()

sign-resolvable RNA 33/38 ATAC 35/38 | <10%-detected landing cells RNA 49 ATAC 7 (of 228)
naming 4: PHLDB1, PHGDH, PLLP, BCAS1
[zmode] RNA s0 = 0.1476 | ATAC s0 = 0.07267
  RNA block off-scale triangles 2 (of which reference column COP/NFOL 2) | |z|>2.0 and not greyed
  ATAC block off-scale triangles 1 (of which reference column COP/NFOL 1) | |z|>2.0 and not greyed
legend handed to the shared band (PANEL_LEGEND=0)
size 7.45 x 5.75 | text right 7.09 (limit 7.25) | bottom 5.60 | non-black text []


## 6. Row 3 legend, row 4 — panels F and G

The legend is re-rendered at the solved column width so its canvas equals its ink and it can share
row 3's top edge. Its `off scale` key is drawn the way the mark actually looks on the body — a
1.2 pt **white** triangle at the clipped edge of a saturated cell — because a key that shows a
2.2 pt blue/red triangle is a different glyph from the one the reader is being asked to find.

**F** puts the same OPC-vs-Oligo contrast that Fig 6 A/B states for the own cohort into the three
chemistries, per nucleus, both channels, in one panel: two rows are the channels, three columns
are the chemistries, and the MWU *P* per panel is read from the ledger and *asserted* to equal the
label drawn ("P = " + p_fig), so a rewritten table cannot leave a stale P on the face.

**G** is the panel that used to be Figure 6 D: the 74-trait expression/chromatin Δ1 pair with the
rank columns, re-cut for the supplement by narrowing columns through environment variables only —
zero vector scaling, no font change. It sits in row 4 next to F so that "one nucleus at a time"
and "all 74 traits at once" are read as the same claim at two resolutions.

In [10]:
# ------------------------------------------------------------ assembly/figS6v16_panels.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/figS6v16_panels.py")
# ---- section per-channel boxes of the delivered file ----
CM = 2.54   # shared-kernel hygiene: the D/E panel cells overwrite CM with 1/2.54 (cm->inch inverse); re-asserted per section, a no-op when the whole file runs
CELLDIR = os.environ.get("S6_CELLS", R_OUT + "/D_crosschemistry/tables_log1p_20261104")
CELLF = {"dsciCAT-seq (own)": CELLDIR + "/cells_dsciCAT_seq.csv",
         "10x Multiome": CELLDIR + "/cells_10x.csv",
         "HT-scCAT-seq": CELLDIR + "/cells_HT_scCAT_seq_HT.csv"}
DIRF = {"dsciCAT-seq (own)": ("dsciCAT_seq_rna", "dsciCAT_seq_atac"),
        "10x Multiome": ("10x_rna", "10x_atac"), "HT-scCAT-seq": ("HT_scCAT_seq_HT_rna", "HT_scCAT_seq_HT_atac")}
_C = {}


def ms_scores(tech, chan):
    """Per-nucleus MS scDRS norm_score + the Fig 2 label. The panel only uses these two columns, so the cache is exactly those two;
    when missing, recompute from .score.gz (part of the ~6 GB per-tech per-channel tree) and write back."""
    key = (tech, chan)
    if key in _C:
        return _C[key]
    slug = DIRF[tech][0 if chan == "rna" else 1]
    fc_ = os.path.join(CACHE, "s6f_%s_%s.csv" % (slug, chan))
    if os.path.exists(fc_):
        df = pd.read_csv(fc_).set_index("cell")
        say("[F] %-16s %-4s per-nucleus scores from cache %s (%d nuclei)" % (tech, chan, os.path.basename(fc_), len(df)))
        _C[key] = df
        return df
    rn, an = DIRF[tech]
    base = R_OUT + "/D_crosschemistry/scdrs/" + rn if chan == "rna" else \
        R_OUT + "/D_crosschemistry/scdrs_log1p_20261104/" + an
    fs = sorted(glob.glob(base + "/*/PASS_Multiple_sclerosis.full_score.gz"))
    if not fs:
        raise SystemExit("[F] no .score.gz under %s and no cache %s: export DSCI_WORK_ROOT, "
                         "or run upstream/S6v16_export_caches.py first to build the caches" % (base, fc_))
    df = pd.concat([pd.read_csv(f, sep="\t", index_col=0, low_memory=False) for f in fs])
    df.index.name = "cell"
    lb = pd.read_csv(CELLF[tech]).set_index("cell")["label"]
    df = df[df.index.isin(lb.index)].copy()
    df["label"] = lb.loc[df.index].values
    df[["norm_score", "label"]].to_csv(fc_)
    df = df[["norm_score", "label"]]
    _C[key] = df
    return df


MWU = pd.read_csv(V8 + "/figS6v10_G_opc_oligo_mwu.csv", dtype={"p_fig": str, "p_label": str})
GPT = {"dsciCAT-seq (own)": dict(s=0.9, alpha=0.20, lighten=0.55),
       "10x Multiome": dict(s=1.6, alpha=0.30, lighten=0.0),
       "HT-scCAT-seq": dict(s=1.6, alpha=0.30, lighten=0.0)}


def _dark(c, thr=0.55):
    rgb = np.array(mcolors.to_rgb(c)); lum = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
    return tuple(rgb * (thr / lum)) if lum > thr else tuple(rgb)


def _lighten(c, f):
    rgb = np.array(mcolors.to_rgb(c)); return tuple(rgb * f + (1 - f))


def panel_boxes(chan, title_side):
    fig = plt.figure(figsize=(W_CELL / CM, 4.20 / CM))
    gs = fig.add_gridspec(1, 3, left=0.135, right=0.985, top=1 - 0.62 / 4.60, bottom=0.155,
                          wspace=0.20)
    rng = np.random.default_rng(20261117)
    for ci, tech in enumerate(ARMS):
        ax = fig.add_subplot(gs[0, ci])
        df = ms_scores(tech, chan)
        mo = df[df.label == "Oligo"].norm_score.values; op = df[df.label == "OPC"].norm_score.values
        st = GPT[tech]
        for pos, arr, col in [(0, op, OPC), (0.6, mo, OLIGO)]:
            ax.scatter(pos + rng.normal(0, 0.07, len(arr)), arr, s=st["s"],
                       color=_lighten(col, 1 - st["lighten"]) if st["lighten"] else col,
                       alpha=st["alpha"], linewidths=0, rasterized=True, zorder=1)
        bp = ax.boxplot([op, mo], positions=[0, 0.6], widths=0.5, patch_artist=True,
                        showfliers=False, whis=[2, 98], medianprops=dict(lw=1.3),
                        boxprops=dict(lw=0.8), whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
        for ln in bp["caps"]:
            c = float(np.mean(ln.get_xdata())); y = float(ln.get_ydata()[0])
            ln.set_data([c - 0.13, c + 0.13], [y, y])
        for b, col in zip(bp["boxes"], [OPC, OLIGO]):
            b.set_facecolor(col); b.set_alpha(0.30); b.set_edgecolor(_dark(col)); b.set_zorder(2)
        for ln, col in zip(bp["medians"], [OPC, OLIGO]):
            ln.set_color(_dark(col)); ln.set_zorder(3)
        for key in ("whiskers", "caps"):
            for ln in bp[key]:
                ln.set_color(OLIGO if ln.get_xdata()[0] > 0.2 else OPC); ln.set_zorder(2)
        r = MWU[(MWU.tech == ARM_LAB[tech]) & (MWU.channel == chan)].iloc[0]
        # Ledger consistency: the face label must be exactly p_fig's "P = " form. If the table is ever rewritten without p_label following,
        # this explodes immediately - no silent "P on the face disagrees with Table S6" drift.
        assert str(r.p_label) == "P = " + str(r.p_fig), (tech, chan, r.p_label, r.p_fig)
        ax.text(0.97, 0.05, str(r.p_label), fontsize=FS_LAB, ha="right", va="bottom",
                color="black", transform=ax.transAxes)
        ax.set_ylim(-1.62, 1.4); ax.set_xlim(-0.5, 1.1)
        ax.axhline(0, color="0.80", lw=0.5, zorder=1)
        ax.set_xticks([0, 0.6]); ax.set_xticklabels(["OPC", "Oligo"], fontsize=FS_LAB)
        ax.set_yticks([-1, 0, 1])
        if ci:
            ax.set_yticklabels([])
        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)
        for sp in ["left", "bottom"]:
            ax.spines[sp].set_linewidth(0.5)
        ax.tick_params(width=0.4, length=1.8, pad=0.8, labelsize=FS_LAB)
        ax.set_title(ARM_LAB[tech], fontsize=FS_TITLE, pad=2.0)
        if ci == 0:
            ax.set_ylabel("per-cell norm_score, %s" % ("RNA" if chan == "rna" else "ATAC"),
                          fontsize=FS_TITLE, labelpad=1.5)
    return savefig(fig, "figS6v16_%s_%sboxes" % (title_side, "RNA" if chan == "rna" else "ATAC"))



# ---- section F combined of the delivered file ----
def _boxes_cell(ax, tech, chan, rng):
    df = ms_scores(tech, chan)
    mo = df[df.label == "Oligo"].norm_score.values; op = df[df.label == "OPC"].norm_score.values
    st = GPT[tech]
    for pos, arr, col in [(0, op, OPC), (0.6, mo, OLIGO)]:
        ax.scatter(pos + rng.normal(0, 0.07, len(arr)), arr, s=st["s"],
                   color=_lighten(col, 1 - st["lighten"]) if st["lighten"] else col,
                   alpha=st["alpha"], linewidths=0, rasterized=True, zorder=1)
    bp = ax.boxplot([op, mo], positions=[0, 0.6], widths=0.5, patch_artist=True,
                    showfliers=False, whis=[2, 98], medianprops=dict(lw=1.3),
                    boxprops=dict(lw=0.8), whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
    for ln in bp["caps"]:
        c = float(np.mean(ln.get_xdata())); y = float(ln.get_ydata()[0])
        ln.set_data([c - 0.13, c + 0.13], [y, y])
    for b, col in zip(bp["boxes"], [OPC, OLIGO]):
        b.set_facecolor(col); b.set_alpha(0.30); b.set_edgecolor(_dark(col)); b.set_zorder(2)
    for ln, col in zip(bp["medians"], [OPC, OLIGO]):
        ln.set_color(_dark(col)); ln.set_zorder(3)
    for key in ("whiskers", "caps"):
        for ln in bp[key]:
            ln.set_color(OLIGO if ln.get_xdata()[0] > 0.2 else OPC); ln.set_zorder(2)
    r = MWU[(MWU.tech == ARM_LAB[tech]) & (MWU.channel == chan)].iloc[0]
    assert str(r.p_label) == "P = " + str(r.p_fig), (tech, chan, r.p_label, r.p_fig)
    # The P label sits lower-right, but Oligo's lower whisker runs right through the digits (verify_text_on_data threshold 6%,
    # which does not fire; by eye it reads as "1.8|e-18"). The white pad hides the whisker - a whisker is a continuous quantity, no misreading.
    ax.text(0.97, 0.05, str(r.p_label), fontsize=FS_LAB, ha="right", va="bottom",
            color="black", transform=ax.transAxes, zorder=7,
            bbox=dict(facecolor="white", edgecolor="none", pad=0.8))
    ax.set_ylim(-1.62, 1.4); ax.set_xlim(-0.5, 1.1)
    ax.axhline(0, color="0.80", lw=0.5, zorder=1)
    ax.set_xticks([0, 0.6]); ax.set_xticklabels(["OPC", "Oligo"], fontsize=FS_LAB)
    ax.set_yticks([-1, 0, 1])
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    for sp in ["left", "bottom"]:
        ax.spines[sp].set_linewidth(0.5)
    ax.tick_params(width=0.4, length=1.8, pad=0.8, labelsize=FS_LAB)


def panel_F_combined():
    """F: RNA and ATAC merged into ONE cell (author 2026-11-17: the narrative first reads "MS risk nucleus by nucleus",
    the two channels are two sides of one thing; splitting into F and G would read one thing as two).
    2 rows = channels, 3 columns = chemistries; column names only on the top row, channel names in a two-line ylabel."""
    H0 = 6.00
    fig = plt.figure(figsize=(W_CELL / CM, H0 / CM))
    gs = fig.add_gridspec(2, 3, left=0.148, right=0.985, top=1 - 0.50 / H0, bottom=0.070,
                          hspace=0.44, wspace=0.20)
    rng = np.random.default_rng(20261117)
    for ri, chan in enumerate(["rna", "atac"]):
        for ci, tech in enumerate(ARMS):
            ax = fig.add_subplot(gs[ri, ci])
            _boxes_cell(ax, tech, chan, rng)
            if ri == 0:
                ax.set_title(ARM_LAB[tech], fontsize=FS_TITLE, pad=2.0)
            if ci == 0:
                ax.set_ylabel("per-cell norm_score\n%s" % ("RNA" if chan == "rna" else "ATAC"),
                              fontsize=FS_TITLE, labelpad=1.5, linespacing=1.30)
            else:
                ax.set_yticklabels([])
    return savefig(fig, "figS6v16_F_cellboxes")

panel_F_combined()

  figS6v16_F_cellboxes.pdf       8.08 x 5.75 cm | ink right 7.97/8.08 bottom 5.70/5.75 | sizes [6.0, 8.0] | non-bw []


'<repo>/figS6/figS6_out/figS6v16_F_cellboxes.pdf'

In [11]:
# final pass at the solved column width (LEG_W was written by the geometry cell)
# ------------------------------------------------------------ assembly/figS6v18_legend_right.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/figS6v18_legend_right.py")
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""figS6v18_legend_right.py - the VERTICAL legend shared by D/E (author 2026-11-17: DE shift left overall, the legend squeezed into the rightmost column).

Why vertical: the horizontal shared legend band (figS6v17, 17.00 x 1.05) took a whole row, page height 20.11 cm.
The vertical version joins row 3 as its third column, saving 1.05 row height + 0.15 attachment gap = **1.20 cm**.
The price: D/E give up this column's width, so the subtype-column width CW shrinks from 0.55 to ~0.48 (the body still has 6 columns x 2 blocks,
cells only narrow 13%, row height and type size unchanged - house spec ranks added whitespace above shrinking type).

The width is not guessed: every text is measured with fitz at its real width, the longest item fixes LW, then CW is solved back,
so that D + E + 2 x GAP + LW is **exactly 17.00** (the same idea as spreading Fig6's column gaps, SKILL §17.9).

Two font sizes 8.0/6.0; face text pure black (the colour-bar end numbers too); gradient direction conventional (blue− left/bottom, red+ right/top).
"""
import os
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import fitz

HERE = os.path.dirname(os.path.abspath(__file__))
# Default = three levels above the repo GitHub/figS6/assembly (the analysis-tree root); override with DSCI_WORK_ROOT.
WORK = os.environ.get("DSCI_WORK_ROOT", os.path.abspath(os.path.join(HERE, "..", "..", "..")))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from arial_style import use_arial, use_two_tier

PAN = os.environ.get("S6_OUT", WORK + "/figures/routeA_out/P0_8C/panels"); os.makedirs(PAN, exist_ok=True)
CM = 2.54
BLACK, GOLD, GREY, EDGE = "#000000", "#b8860b", "#d9d9d9", "#333333"
ZCLIP = float(os.environ.get("PANEL_ZCLIP", 2.0))
LW = float(os.environ.get("LEG_W", 2.00))          # canvas width (>= what is needed; tight trims to the ink width)
LH = float(os.environ.get("LEG_H", 5.75))          # matches the D/E row height so they share a top-aligned row
YCENTER = float(os.environ.get("LEG_YC", 3.38))    # the stack's vertical centre: aligns with the body mid-line (1.35+5.41)/2 of both cells
CMAP = LinearSegmentedColormap.from_list("by", ["#2166ac", "#f7f7f7", "#b2182b"])
use_arial(); use_two_tier()


def tw(t, fs=6.0):
    return fitz.get_text_length(t, fontname="helv", fontsize=fs) * CM / 72.0


# At STEP 0.36 a two-line 6pt key (0.42 together) would touch itself; raised to 0.44; stack height 3.97,
# still centred on the body mid-line 3.38 (1.40-5.37, inside the 5.75 page).
BARW, SWH, GAPX, STEP = 0.22, 0.15, 0.07, 0.44
# The off-scale key's swatch size and triangle: MS_OFF is the same 1.2 as the body triangles in the panel scripts,
# not "a bit bigger in the legend" - enlarging would lie: a reader measuring the legend against the face could not match the cells.
CHW, CHH, MS_OFF = 0.16, 0.11, 1.2
TITLE, TBAR = "z (per gene)", 1.60
KEYS = [("sw", "not detected"), ("car", "off scale"),
        ("bar", "tested\n(MOL6)"), ("bar", "reference\n(COP/NFOL)")]
# per-item width = swatch + gap + the widest text line
WID = {"sw": BARW + GAPX + tw("not detected"), "car": 0.34 + GAPX + tw("off scale"),
       "bar": None}
wbar = [BARW + GAPX + max(tw(l) for l in t.split("\n")) for k, t in KEYS if k == "bar"]
wcol = BARW + GAPX + tw("+%.1f" % ZCLIP)
tot = max(wcol, wbar[0], wbar[1], tw(TITLE), BARW + GAPX + tw("not detected"),
          0.34 + GAPX + tw("off scale"))
print("legend needs width %.2f cm (longest item = %s)" % (tot, ["colour scale", "not detected", "off scale",
      "tested(MOL6)", "reference(COP/NFOL)", "title"][
          [wcol, BARW + GAPX + tw("not detected"), 0.34 + GAPX + tw("off scale"),
           wbar[0], wbar[1], tw(TITLE)].index(tot)]))
if LW < tot + 0.10:
    raise SystemExit("LEG_W=%.2f does not fit the longest item %.2f (needs >= %.2f)" % (LW, tot, tot + 0.10))

fig = plt.figure(figsize=(LW / CM, LH / CM))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, LW); ax.set_ylim(0, LH); ax.axis("off")
X0 = 0.10
stack = 0.21 + 0.10 + TBAR + 0.30 + STEP * len(KEYS)      # title + colour scale + gap + four keys
y = YCENTER + stack / 2.0                                   # stack from the top down
ax.text(X0, y - 0.21, TITLE, fontsize=6.0, ha="left", va="bottom", color=BLACK)
y -= 0.21 + 0.10
ax.imshow(np.linspace(-ZCLIP, ZCLIP, 64)[:, None],
          extent=(X0, X0 + BARW, y - TBAR, y), cmap=CMAP, aspect="auto", zorder=3)
ax.add_patch(Rectangle((X0, y - TBAR), BARW, TBAR, fc="none", ec=EDGE, lw=0.3))
for dy, lab in ((0.0, "+%.1f" % ZCLIP), (TBAR / 2.0, "0"), (TBAR, "−%.1f" % ZCLIP)):
    ax.text(X0 + BARW + GAPX, y - dy, lab, fontsize=6.0, ha="left", va="center", color=BLACK)
y -= TBAR + 0.30
for kind, t in KEYS:
    yy = y - STEP / 2.0
    if kind == "sw":
        ax.add_patch(Rectangle((X0, yy - SWH / 2), BARW, SWH, fc=GREY, ec="#8c8c8c", lw=0.3))
    elif kind == "car":
        # The key must look exactly like the face: a WHITE triangle ms=1.2 stuck to the off-scale cell's left edge (negative) / right edge (positive),
        # on the saturated blue/red clipped at ±ZCLIP. This used to draw a 2.2 pt blue/red triangle -
        # matching the face in neither colour nor size (a reader hunting for blue triangles finds none).
        # Now a "swatch + same-size white triangle": swatch width 0.16, the two together still 0.34, so the legend width is unchanged and CW needs no re-solving.
        for k, (cx, fc, mk, mx) in enumerate((
                (X0, CMAP(0.0), "<", 0.030),                 # negative off-scale: blue swatch + white < on the left edge
                (X0 + 0.18, CMAP(1.0), ">", 0.130))):        # positive off-scale: red swatch + white > on the right edge
            ax.add_patch(Rectangle((cx, yy - CHH / 2), CHW, CHH, fc=fc, ec="none"))
            ax.plot(cx + mx, yy, marker=mk, ms=MS_OFF, color="#ffffff", mew=0.3, zorder=6)
    else:
        ax.add_patch(Rectangle((X0, yy - 0.038), BARW, 0.075,
                               fc=GOLD if "tested" in t else BLACK, ec="none"))
    lines = t.split("\n")
    for k, ln in enumerate(lines):        # two-line keys: centred on y as a block
        ax.text(X0 + (BARW if kind != "car" else 0.34) + GAPX,
                yy + (0.09 if len(lines) > 1 and k == 0 else (-0.09 if len(lines) > 1 else 0.0))
                - 0.0, ln, fontsize=6.0, ha="left", va="center", color=BLACK)
    y -= STEP
print("  stack %.2f-%.2f, centre %.2f (target %.2f)" % (y, y + stack, y + stack / 2.0, YCENTER))
out = PAN + "/figS6v18_DE_legend_v.pdf"
fig.savefig(out, dpi=600)
d = fitz.open(out); p = d[0]; ws = p.get_text("words")
sp = [s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]]
print("  %-30s page %.2f x %.2f cm | text right %.2f bottom %.2f top %.2f | sizes %s | non-bw %s"
      % (os.path.basename(out), p.rect.width * CM / 72, p.rect.height * CM / 72,
         max(w[2] * CM / 72 for w in ws), max(w[3] * CM / 72 for w in ws),
         min(w[1] * CM / 72 for w in ws), sorted({round(s["size"], 1) for s in sp}),
         [{"#%06x" % s["color"]} for s in sp if s["color"] not in (0, 16777215)]))
d.close()

legend needs width 1.51 cm (longest item = reference(COP/NFOL))
  stack 1.40-5.37, centre 3.38 (target 3.38)
  figS6v18_DE_legend_v.pdf       page 1.62 x 5.75 cm | text right 1.61 bottom 4.32 top 0.36 | sizes [6.0] | non-bw []


In [12]:
os.environ.update(COMPACT="1", SUF="_s6v16G", WD="0.40", WK="0.40", WG="0.12",
                  LM="1.55", GAPC="0.42", RMC="1.10", TABD=TABD, OUTDIR=OUTD)
# ------------------------------------------------------------ assembly/make_figS6v16_G_heatmap.py (verbatim)
# The cell body below IS the delivered pipeline script. `__file__` is injected so
# the script's own HERE/WORK resolution is identical to running it as a file.
# Everything after the CLI entry point (`if __name__ == "__main__"`) is dropped:
# one kernel runs many cells, so a script that re-launches its sibling panels would
# redraw them under leaked rcParams. The panel functions are called explicitly.
__file__ = os.path.join(REPO, "assembly/make_figS6v16_G_heatmap.py")
# -*- coding: utf-8 -*-
"""Fig 6 · D-panel heatmap version (v21): one RNA and one ATAC heatmap, side by side.
 rows = the 74 traits (row order = ascending mean of the three chemistries' rank_rna_adj; only the expression channel orders, chromatin does not)
 each heatmap has 6 columns = 3 chemistries x [Δ1 column | rank column], with narrow gaps between chemistries
   column layout (8 slots): Δ0 rk0 | gap | Δ1 rk1 | gap | Δ2 rk2  - data column indices POS = (0,1)(3,4)(6,7)
 Δ1 column = within-column z score (coolwarm, clipped symmetrically at ±LIM) - Δ1 absolutes are never compared across chemistries, hence the within-column standardization
 rank column = the rank itself (single-hue sequential, rank 1 = darkest); ranks are the only thing allowed to be compared across chemistries
Only the three traits of figure C are labelled (MS / SCZ / HEIGHT); the other 71 rows carry no text.
This version follows the author's notes:
 - the grey "ladder" line across the top is deleted; content moves up (ρ/pass verdicts live only in the caption and the tables)
 - titles become regular Arial 8pt black: left "Gene expression", right "Gene score"
 - the three chemistry names move to the bottom of the heatmap, in full: dsciCAT-seq / 10xMultiome / HT-scCAT-seq
 - the small "Δ / rk" legend line is deleted (SUBRK=1 restores it) - the two columns' meanings are carried by the right colour-bar titles
 - every text in the figure is regular Arial 8pt black - no greys, no bold (in-cell ranks 6pt, column width only 0.52 cm)
Every number is read from tables/*.csv; nothing is recomputed in the figure (ρ/n are only formatted).
Environment: RKCMAP rank colour map / RKBIN rank binning levels (0=continuous); COMPACT=1 the main-figure compact row-pitch version; WD/WK/WG/RH column widths and row height (cm); LM/GAPC/RMC/TOP/BOT margins (cm);
 VERBOSE=1 adds n; MARKRANKS=1 adds #rna->#atac on the right; SUBRK=1 restores the Δ/rk small line; SUF output-name suffix; PNGOUT writes a PNG.
"""
import os, sys
# ---- portability (2026-11-18, GitHub cleanup): <work> was hard-coded here and would not run in the repo.
#       HERE = this script's directory; WORK = the analysis-tree root (DSCI_WORK_ROOT, inferred from the repo location by default).
HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("DSCI_WORK_ROOT", os.path.abspath(os.path.join(HERE, "..", "..", "..")))
sys.path.insert(0, HERE)              # fig6_palette.py ships in the same directory as this script
if os.environ.get("DSCI_PYLIBS"):
    sys.path.insert(0, os.environ["DSCI_PYLIBS"])
sys.path.insert(0, os.path.join(WORK, "scripts", "MS_GWAS", "fig6_repro", "deps"))
import numpy as np, pandas as pd
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
from matplotlib import font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

FDIR = os.environ.get("DSCI_FONTS", os.path.join(WORK, "fonts"))
for f in ("ARIAL.TTF", "ARIALBD.TTF"):
    if os.path.exists(os.path.join(FDIR, f)):
        fm.fontManager.addfont(os.path.join(FDIR, f))
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams.update({"axes.linewidth": 0.55, "font.size": 8.0})
CM = 2.54
_SHIPPED = os.path.join(os.path.dirname(HERE), "tables_v16")     # figS6/tables_v16
TABD = os.environ.get("TABD", _SHIPPED if os.path.exists(os.path.join(_SHIPPED, "fig6D_delta1_74traits_per_tech.csv"))
                       else os.path.join(WORK, "figures/routeA_out/D_crosschemistry/tables"))
OUT = os.environ.get("OUTDIR", os.environ.get("S6_OUT", os.path.join(WORK, "FY/oligo/fig6_rework_v2")))
COMPACT = os.environ.get("COMPACT", "0") == "1"
os.makedirs(OUT, exist_ok=True)

D = pd.read_csv(TABD + "/fig6D_delta1_74traits_per_tech.csv")
TECH = ["dsciCAT-seq (own)", "10x Multiome", "HT-scCAT-seq"]
NAME = ["dsciCAT\nseq", "10x\nMultiome", "HT-scCAT\nseq"]      # bottom full names (author-specified); one 8pt horizontal line would need 1.95 cm per group,
#  so D would grow to 15.5 cm and leave the main-figure row; hence two lines; at a 1.26 cm group pitch the widest line "Multiome" (1.16 cm) never overlaps
NAME1L = ["dsciCAT-seq", "10xMultiome", "HT-scCAT-seq"]
SHORT = ["dsciCAT-seq", "10x", "HT"]
from fig6_palette import TRAIT_COLOR   # shared with C: MS red / SCZ dark blue / HEIGHT grey
import matplotlib.patheffects as pe
# Submission hard gate: every visible glyph >= 5.5 pt (scripts/final_audit.py:206). The author asked to "shrink a bit",
# 5.0 pt would fit the column but break this gate, so the shrink stops at 5.5 pt (rank column 0.55 cm; a two-digit 5.5 pt number needs 0.22 cm, still comfortable).
FSR = float(os.environ.get("FSR", "5.5"))          # in-cell rank size (was 6.0; the author asked to shrink further; the 5.5 pt hard gate sets the floor)
# No path_effects: measured - the PDF backend then writes text as vector outlines, invisible to get_text()
# (the "every face number traces to a table row" acceptance gate would fail on the spot). Instead the same number is printed twice: a larger white layer underneath,
# then the colour layer: both are real text, white hugs only the glyph outline, and unlike a white box it does not hide the heatmap's thin cells.
FSW = float(os.environ.get("FSW", "6.8"))          # white underlay size = FSR + 1.3, ~0.65 pt visible all around
MARK = {"PASS_Multiple_sclerosis": "MS",
        "PASS_Schizophrenia_Pardinas2018": "SCZ",
        "UKB_460K.body_HEIGHTz": "HEIGHT"}
# ---- three placements for the rank digits (author 11-03 question: raw digits on the cell are illegible; a white backing is legible but hides the heatmap) ----
# The root cause is geometry: the delivered row height 0.058 cm = 1.64 pt, while the hard gate demands >= 5.5 pt type - the glyph is inevitably taller than the row,
# so "in the cell" and "not covering neighbouring rows" cannot both hold at equal row heights; only geometry or placement can change.
#   halo  = current: equal row heights, colour glyph + enlarged white underlay (presses on ~1 row above and below)
#   cell  = the 3 named rows grow to TALLH cm and the other 71 shrink proportionally (total heatmap height unchanged);
#           the named rows' rank cells stay blank and the digit prints inside its own row - no neighbouring row touched, no white underlay needed
#   none  = no digits: the heatmap untouched, all ranks go to the caption
NUMMODE = os.environ.get("NUMMODE", "none")   # author 11-03, seventh revision, decided: no ranks on the face; rankings go into the text (halo/cell kept as fallback)
TALLH = float(os.environ.get("TALLH", "0.22"))      # cell mode: physical height of a named row (cm, = 6.2 pt)
FSD = float(os.environ.get("FSD", "6.0"))           # cell mode: rank size (tall enough rows, larger than halo's 5.5)
# In cell mode rows are stretched; if the Δ cell stretched along, it would become a 4x-area block (area = visual weight, easily read as "bigger effect").
# KEEPD=1: a named row's Δ cell keeps the regular row height (centred), blank above and below - colour and area unchanged; only a reading strip is inserted into the row.
KEEPD = os.environ.get("KEEPD", "1") == "1"
FS = 8.0                                                        # size of every annotation in the figure (author: regular Arial 8pt black)

# ---- row order: only the expression channel (the reference channel) orders; chromatin does not participate ----
rk = D.pivot(index="trait", columns="tech", values="rank_rna_adj")[TECH]
order = rk.mean(axis=1).sort_values().index.tolist()
P = {t: D[D.tech == t].set_index("trait").loc[order] for t in TECH}
NT = len(order)

# ---- column layout ----
W_D = float(os.environ.get("WD", 0.55 if COMPACT else 0.92))   # Δ column width (cm)
W_K = float(os.environ.get("WK", 0.55 if COMPACT else 0.92))   # rank column width
W_G = float(os.environ.get("WG", 0.16 if COMPACT else 0.26))   # chemistry gap
RH = float(os.environ.get("RH", 0.058 if COMPACT else 0.112))  # row height (cm)
# Row height became "per-row physical height" so named rows can grow alone without changing the total heatmap height (the y axis then runs in cm, not rows)
MARKED = [i for i, tr in enumerate(order) if tr in MARK]
if NUMMODE == "cell" and MARKED:
    base = (NT * RH - len(MARKED) * TALLH) / (NT - len(MARKED))
    assert base > 0, "TALLH too large: the remaining rows would go negative"
    HROW = np.where(np.isin(np.arange(NT), MARKED), TALLH, base)
else:
    HROW = np.full(NT, RH)
YEDGE = np.concatenate([[0.0], np.cumsum(HROW)])     # row edges (cm)
YC = (YEDGE[:-1] + YEDGE[1:]) / 2.0                  # row centres (cm)
BASEH = float(np.min(HROW))                          # the regular row height (named Δ cells use this height under KEEPD)
xs = []
for i in range(3):
    xs += [W_D, W_K]
    if i < 2:
        xs.append(W_G)
NC = len(xs)                              # 8
xedge = np.concatenate([[0.0], np.cumsum(xs)])
XCEN = (xedge[:-1] + xedge[1:]) / 2.0
POS = [(3 * i, 3 * i + 1) for i in range(3)]        # (Δ column index, rank column index)
assert POS[2][1] == NC - 1, POS

# ---- layout: explicit cm placement. gridspec wspace is a column-width fraction; no margin you compute can fit an 8pt colour-bar title, so it inevitably spills into the heatmap ----
LM = float(os.environ.get("LM", 2.05 if COMPACT else 2.30))     # left trait names
GAPC = float(os.environ.get("GAPC", 0.60 if COMPACT else 0.80))  # between the two heatmaps
RMC = float(os.environ.get("RMC", 1.45 if COMPACT else 1.55))    # right colour bars
TOP = float(os.environ.get("TOP", 0.46))                          # top titles
BOT = float(os.environ.get("BOT", 0.68 if COMPACT else 0.72))      # bottom chemistry names (two lines of 8pt)
TW = 2 * xedge[-1] + LM + GAPC + RMC
TH = YEDGE[-1] + TOP + BOT
fig = plt.figure(figsize=(TW / CM, TH / CM), dpi=300)


def axpos(x0):
    return [x0 / TW, BOT / TH, xedge[-1] / TW, (TH - TOP - BOT) / TH]


def z(a):
    a = np.asarray(a, float)
    return (a - a.mean()) / a.std(ddof=0)


LIM, RMAX = 1.40, float(NT)
CD = mpl.colormaps["coolwarm"].copy(); CD.set_bad("none")
# ---- rank colour map (author 11-03: with the digits off the face, colours must convey “roughly how different”) ----
# The semantic stays: DARKER = BETTER rank (rank 1 darkest), so all built-in ramps use the `_r` variants.
# Why measured rather than “just pick something nice”: 74 rows in 4.29 cm - the colour difference between adjacent rows is all the information available.
# Sampling the ramp into 74 levels and computing adjacent-pair Lab ΔE (JND ~ 2.3): the current greyblue has median **1.09** and **0%** of pairs above JND -
# — “can't tell them apart” is a necessity, not a taste issue. Candidates: cividis_r 2.13 / viridis_r 3.13 / inferno_r 3.61.
# `RKBIN>0` additionally bins the 74 ranks (staircase): adjacent distinguishability comes from colour JUMPS instead of fine differences, at the cost of within-bin order.
RK_STOPS = {"greyblue": ["#F4F6F8", "#8FA0B8", "#1B2A45"],      # the previously delivered ramp (median adjacent ΔE 1.09)
            "ink": ["#FFFFFF", "#000000"],                       # pure grey, ΔE 1.39, no better than the current
            "cividis_r": "cividis_r", "viridis_r": "viridis_r", "inferno_r": "inferno_r",
            "magma_r": "magma_r"}
RKCMAP = os.environ.get("RKCMAP", "cividis_r")
RKBIN = int(os.environ.get("RKBIN", "0"))
_rk = RK_STOPS[RKCMAP]
CR = (mpl.colormaps[_rk].copy() if isinstance(_rk, str)
      else LinearSegmentedColormap.from_list("rk", _rk))
CR.set_bad("none")
if RKBIN:
    CR = CR.resampled(RKBIN)
    CRNORM = mpl.colors.BoundaryNorm(np.linspace(0, 1, RKBIN + 1), RKBIN)
else:
    CRNORM = mpl.colors.Normalize(0, 1)

for ch, ttl, ax in (("rna", "Gene expression", fig.add_axes(axpos(LM))),
                    ("atac", "Gene score", fig.add_axes(axpos(LM + xedge[-1] + GAPC)))):
    Cd = np.full((NT, NC), np.nan)
    Ck = np.full((NT, NC), np.nan)
    key = "d1_rna" if ch == "rna" else "d1_atac"
    rc = "rank_rna_adj" if ch == "rna" else "rank_atac_adj"
    for i, t in enumerate(TECH):
        Cd[:, POS[i][0]] = z(P[t][key].to_numpy())
        Ck[:, POS[i][1]] = 1.0 - (P[t][rc].to_numpy() - 1) / (RMAX - 1)
    if RKBIN:
        Ck[:, [POS[i][1] for i in range(3)]] = (np.floor(Ck[:, [POS[i][1] for i in range(3)]] * RKBIN)
                                                .clip(0, RKBIN - 1) + 0.5)   # bin centres, aligned with CRNORM
    if NUMMODE == "cell":
        # the named rows' rank cells stay blank: the digit is that cell's information; yielding to it loses nothing,
        # and without the white underlay no white block ever covers neighbouring rows (the Δ columns keep their colours intact)
        Ck[np.ix_(MARKED, [POS[i][1] for i in range(3)])] = np.nan
        if KEEPD:
            ZD = Cd.copy()
            Cd[np.ix_(MARKED, [POS[i][0] for i in range(3)])] = np.nan
    Xe, Ye = np.meshgrid(xedge, YEDGE)
    ax.pcolormesh(Xe, Ye, Cd, cmap=CD, vmin=-LIM, vmax=LIM, shading="flat", zorder=2)
    ax.pcolormesh(Xe, Ye, Ck, cmap=CR, norm=CRNORM, shading="flat", zorder=2)
    if NUMMODE == "cell" and KEEPD:
        # the named Δ cells are redrawn at the regular row height (colours still mapped by coolwarm over ±LIM, consistent with the other rows)
        for r in MARKED:
            for i in range(3):
                c = POS[i][0]
                ax.add_patch(Rectangle((xedge[c], YC[r] - BASEH / 2), xs[c], BASEH,
                                       facecolor=CD((np.clip(ZD[r, c], -LIM, LIM) + LIM) / (2 * LIM)),
                                       edgecolor="none", zorder=2.5))
    ax.set_xlim(0, xedge[-1]); ax.set_ylim(YEDGE[-1], 0.0)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    # each chemistry's (Δ, rank) pair gets a shared frame so the eye groups the two columns; chemistries separate by the narrow gap only
    for i in range(3):
        x0, x1 = xedge[POS[i][0]], xedge[POS[i][1] + 1]
        ax.add_patch(Rectangle((x0, YEDGE[0]), x1 - x0, YEDGE[-1] - YEDGE[0], fill=False, ec="#666666", lw=0.5,
                               clip_on=False, zorder=4))
    ax.add_patch(Rectangle((0, YEDGE[0]), xedge[-1], YEDGE[-1] - YEDGE[0], fill=False, ec="#333333", lw=0.7, clip_on=False))
    # offsets are converted to row units by physical height: when row heights change nothing crowds (compact was once caught with 2 overlaps)
    if os.environ.get("SUBRK") == "1":
        y_rk = -0.05
        for i in range(3):
            ax.text(XCEN[POS[i][0]], y_rk, "\u0394", ha="center", va="bottom", fontsize=FS, color="black")
            ax.text(XCEN[POS[i][1]], y_rk, "rk", ha="center", va="bottom", fontsize=FS, color="black")
        ax.set_title(ttl, fontsize=FS, color="black", pad=3 + 8 * FS / 72.0 * 2.83)
    else:
        ax.set_title(ttl, fontsize=FS, color="black", pad=3)
    # bottom chemistry names: a group pitch of only W_D+W_K+W_G ~ 1.2 cm - 8pt full names on one line inevitably overlap (measured 12-15 pt overlap),
    # so they are rotated 90°. Horizontal would need ~0.88 cm columns and grow D to 15 cm, over the main-figure row.
    y_tech = YEDGE[-1] + 0.09
    rot = os.environ.get("TECHROT", "0")
    for i in range(3):
        cx = (xedge[POS[i][0]] + xedge[POS[i][1] + 1]) / 2
        if rot == "90":
            ax.text(cx, y_tech, NAME1L[i], ha="center", va="top", rotation=90, fontsize=FS, color="black")
        else:
            ax.text(cx, y_tech, NAME[i], ha="center", va="top", fontsize=FS, color="black", linespacing=1.12)

# ---- only the three figure-C traits are labelled: left name + short dash; in-cell ranks use a small white box (row height 0.06 cm; a full-row frame would double up) ----
for tr, nm in MARK.items():
    if tr not in order:
        continue
    r = order.index(tr)
    for ch, ax in (("rna", fig.axes[0]), ("atac", fig.axes[1])):
        if NUMMODE == "none":
            break
        for i in range(3):
            v = int(P[TECH[i]]["rank_rna_adj" if ch == "rna" else "rank_atac_adj"].iloc[r])
            # author 11-03: rank digits take the trait colour, smaller, white box removed (it hid the heatmap's thin cells).
            # no backing at all, but a white outline via path effect: without it, #D51F26's "11"
            # would vanish on a dark-red cell - the outline costs 0.9 pt of glyph edge, hides no colour, and keeps the digit legible on any cell.
            _cx = (xedge[POS[i][1]] + xedge[POS[i][1] + 1]) / 2
            if NUMMODE == "cell":
                ax.text(_cx, YC[r], str(v), ha="center", va="center", fontsize=FSD,
                        color=TRAIT_COLOR[nm], zorder=7)
            else:
                ax.text(_cx, YC[r], str(v), ha="center", va="center", fontsize=FSW, color="white", zorder=6.5)
                ax.text(_cx, YC[r], str(v), ha="center", va="center", fontsize=FSR, color=TRAIT_COLOR[nm], zorder=7)
    # named traits / dashes use the same trait colours as figure C (fig6_palette is the single source; the acceptance script compares the printed ink on the PDF)
    tc = TRAIT_COLOR[nm]
    fig.axes[0].plot([-0.42, -0.16], [YC[r], YC[r]], color=tc, lw=1.0, clip_on=False, zorder=7)
    fig.axes[0].text(-0.50, YC[r], nm, ha="right", va="center", fontsize=FS, color=tc)
    if os.environ.get("MARKRANKS") == "1":
        fig.axes[1].text(xedge[-1] + 0.22, YC[r], "  ".join(
            "#%d\u2192#%d" % (P[t].rank_rna_adj.iloc[r], P[t].rank_atac_adj.iloc[r]) for t in TECH),
            ha="left", va="center", fontsize=FS, color="black")

# ---- right colour bars: two vertical bars stacked, ticks right-aligned, titles horizontal above (side-by-side 8pt titles would necessarily spill into the heatmap) ----
bx = (LM + 2 * xedge[-1] + GAPC + 0.42) / TW
bw = 0.13 / TW
for y0, h0, cmap, lab, ticks, fmt in (
        (0.575, 0.245, CD, "\u03941 (z)", [-LIM, 0, LIM], lambda v, p: "%+.1f" % v),
        (0.185, 0.245, CR, "Rank", [0, 0.5, 1], None)):
    cax = fig.add_axes([bx, y0, bw, h0])
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=(CRNORM if cmap is CR else
                                                mpl.colors.Normalize(-LIM, LIM)))
    sm.set_array([-LIM, LIM] if cmap is CD else [0, 1])
    mpl.colorbar.ColorbarBase(cax, mappable=sm, orientation="vertical", ticks=ticks)
    cax.yaxis.set_ticks_position("right"); cax.yaxis.set_label_position("right")
    if fmt:
        cax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(fmt))
    else:
        cax.set_yticklabels(["74", "37", "1"])
    cax.tick_params(labelsize=FS, length=1.6, width=0.5, pad=1.6)
    for s in cax.spines.values():
        s.set_linewidth(0.5)
    cax.text(0.5, 1.19, lab, transform=cax.transAxes, ha="center", va="bottom",
             fontsize=FS, color="black")

if os.environ.get("VERBOSE") == "1":
    LAD = pd.read_csv(TABD + "/fig6D_crosschem_ladder.csv").set_index("tech")
    for i, t in enumerate(TECH):
        fig.text(0.0075, 0.93 - 0.045 * i, "n %s: %s / %s" % (SHORT[i], f"{int(LAD.loc[t,'n_oligo']):,}",
                 f"{int(LAD.loc[t,'n_opc']):,}"), fontsize=FS, color="black", va="top", ha="left")

# ---- self-check: the boxed ranks and the Δ columns' z must trace digit-by-digit into the delivered tables ----
for tr, nm in MARK.items():
    if tr not in order:
        continue
    r = order.index(tr)
    chk = []
    for i, t in enumerate(TECH):
        dr, da = P[t].d1_rna.iloc[r], P[t].d1_atac.iloc[r]
        zr = (dr - P[t].d1_rna.mean()) / P[t].d1_rna.std(ddof=0)
        za = (da - P[t].d1_atac.mean()) / P[t].d1_atac.std(ddof=0)
        kr, ka = int(P[t].rank_rna_adj.iloc[r]), int(P[t].rank_atac_adj.iloc[r])
        assert abs(dr - float(D[(D.tech == t) & (D.trait == tr)].d1_rna.iloc[0])) < 1e-9, (nm, t, "d1_rna")
        assert abs(da - float(D[(D.tech == t) & (D.trait == tr)].d1_atac.iloc[0])) < 1e-9, (nm, t, "d1_atac")
        assert kr == int(D[(D.tech == t) & (D.trait == tr)].rank_rna_adj.iloc[0]), (nm, t, "rank_rna")
        assert ka == int(D[(D.tech == t) & (D.trait == tr)].rank_atac_adj.iloc[0]), (nm, t, "rank_atac")
        chk.append("%s #%d/#%d (z %+.2f/%+.2f)" % (SHORT[i], kr, ka, zr, za))
    print("[selfcheck] %-7s %s" % (nm, "  ".join(chk)))

suf = ("_compact" if COMPACT else "") + os.environ.get("SUF", "")
out = OUT + "/fig6D_heatmap_pair%s" % suf
fig.savefig(out + ".pdf", bbox_inches="tight", pad_inches=0.03)
if os.environ.get("PNGOUT"):
    fig.savefig(os.environ["PNGOUT"] + "/fig6D_heatmap_pair%s.png" % suf, dpi=300, bbox_inches="tight", pad_inches=0.03)
print("saved %s.pdf | %.2f x %.2f cm nominal | %d rows x %d cols (data %s) x 2 panels | NUMMODE=%s "
      "RKCMAP=%s RKBIN=%d heat=%.2f cm"
      % (out, TW, TH, NT, NC, POS, NUMMODE, RKCMAP, RKBIN, YEDGE[-1]))

[selfcheck] MS      dsciCAT-seq #7/#8 (z +0.89/+0.96)  10x #4/#55 (z +0.96/-0.86)  HT #11/#47 (z +0.64/-0.64)
[selfcheck] SCZ     dsciCAT-seq #63/#66 (z -1.30/-1.57)  10x #63/#46 (z -1.50/-0.43)  HT #67/#72 (z -1.83/-2.22)
[selfcheck] HEIGHT  dsciCAT-seq #33/#42 (z +0.40/+0.07)  10x #40/#24 (z +0.31/+0.68)  HT #47/#20 (z +0.14/+0.84)
saved <repo>/figS6/figS6_out/fig6D_heatmap_pair_compact_s6v16G.pdf | 8.35 x 5.43 cm nominal | 74 rows x 8 cols (data [(0, 1), (3, 4), (6, 7)]) x 2 panels | NUMMODE=none RKCMAP=cividis_r RKBIN=0 heat=4.29 cm


---

## 7. What this notebook produces

Seven panel PDFs + the shared legend, all at the delivered sizes:

| output | size (cm) |
|---|---|
| `figS6_out/figS6v16_A_loci.pdf` | 10.78 x 2.79 |
| `figS6_out/figS6v16_B_perm.pdf` | 5.38 x 3.05 |
| `figS6_out/figS6v16_C_umapbins.pdf` | 17.00 x 3.00 |
| `figS6_out/fig6v32_C_top1000_landing_s6v18D.pdf` | 7.45 x 5.75 |
| `figS6_out/fig6v31_C_landing_s6v18E.pdf` | 7.45 x 5.75 |
| `figS6_out/figS6v18_DE_legend_v.pdf` | 1.62 x 5.75 |
| `figS6_out/figS6v16_F_cellboxes.pdf` | 8.08 x 5.75 |
| `figS6_out/fig6D_heatmap_pair_compact_s6v16G.pdf` | 8.64 x 5.39 |

The four-row page `Figure_S6_v16.pdf` (17.00 x 18.91 cm) is composed from these eight files by
the delivered driver `assembly/figS6v16_run_all.sh` (step 4/5, `figS6v16_compose.py`) — the
composition code is deliberately **not** part of this notebook (author instruction). The
word-level gate against the delivered page lives in `dev/`, not here.